# 🎨 Udacity — Design of Computer Programs
### Peter Norvig


## 🃏 Lesson 1 — How to Write a Poker Program

### 🔍 Outlining the Problem

To write any program, follow 3 phases:
1. **Understand** — inventory the concepts of the problem
2. **Specify** — define it in terms that can become code
3. **Design** — structure it into code

---

### 📦 Concepts Inventory

| Concept | Description |
|---|---|
| **Card** | Has a `rank` (2–Ace) and a `suit` (♠ ♥ ♦ ♣) |
| **Hand** | A collection of 5 cards |
| **Hand Rank** | A value that determines how strong a hand is |
| **poker()** | Takes a list of hands → returns the best one |

---

### 🏆 Hand Rankings (strongest → weakest)

| # | Hand | Description |
|---|---|---|
| 1 | Straight Flush | 5 consecutive cards, same suit |
| 2 | Four of a Kind | 4 cards of the same rank |
| 3 | Full House | 3 of a kind + a pair |
| 4 | Flush | 5 cards of the same suit |
| 5 | Straight | 5 consecutive cards, any suit |
| 6 | Three of a Kind | 3 cards of the same rank |
| 7 | Two Pair | 2 different pairs |
| 8 | One Pair | 2 cards of the same rank |
| 9 | High Card | None of the above |

---

### 🧩 Key Concepts for Hand Rank Logic
- **n-of-a-kind** → n cards of the same rank (e.g. two 2s, two Jacks)
- **Straight** → 5 consecutive ranks (e.g. 5-6-7-8-9)
- **Flush** → 5 cards of the same suit (e.g. all ♠)
- **Ties** → when two hands have the same rank, compare by **card values** (higher card wins)

---

### 🎯 Goal of the `poker()` function

```python
def poker(hands):
    """Takes a list of hands, returns the best hand."""
    return max(hands, key=hand_rank)
```

> 💡 Using `max()` with a `key` function is a clean **functional programming** pattern — no loops needed.


### 🗂️ Representing Hands — Choosing a Data Structure

**Best choice → list of tuples** `[(rank, suit), ...]` ✅

| Option | Example | Why not? |
|---|---|---|
| List of strings | `['JS', 'JD', 'JC']` | Hard to extract rank/suit — need to parse the string |
| **List of tuples** ✅ | `[(11,'S'), (11,'D'), (11,'C')]` | Rank and suit already separated → easy to work with |
| Set of tuples | `{(11,'S'), (11,'D'), ...}` | Unordered — can't rely on card position |

```python
card = (11, 'S')   # Jack of Spades
card[0]            # → 11  (rank)
card[1]            # → 'S' (suit)
```


### 🎯 Poker Function & Testing

**Goal:** `poker(hands)` takes a list of hands and returns the best one.

```python
def poker(hands):
    "Return the best hand: poker([hand,...]) => hand"
    return max(hands, key=hand_rank)

def hand_rank(hand):
    return None  # placeholder — to be implemented later
```

> 💡 `max()` with `key=hand_rank` picks the hand with the highest rank value. Clean and no loops needed.

---

### ✅ Testing with `assert`

Write a `test()` function with **assert statements** — if an assertion is False, Python stops and raises an error immediately.

```python
def test():
    sf = "6C 7C 8C 9C TC".split()  # Straight Flush
    fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind
    fh = "TD TC TH 7C 7D".split()  # Full House

    assert poker([sf, fk, fh]) == sf   # straight flush beats all
    assert poker([fk, fh]) == fk       # four of a kind beats full house
    assert poker([fh, fh]) == fh       # tie → returns first fh
```

**Why these test cases?**
- Test the **happy path** (sf wins)
- Test a **specific matchup** (fk vs fh)
- Test a **tie** — `max()` returns the first element when values are equal


### 🏅 hand_rank() — Returning a Comparable Value

**Goal:** `hand_rank(hand)` must return a value that allows comparing two hands correctly.

**Problem with returning just a single integer (0–8):**
```python
pair_of_10s → 1
pair_of_9s  → 1   # same rank! but 10s should win
```
→ A single number is **not enough** to break ties within the same rank.

---

### 🔢 Three Options to Fix This

| Option | Four-of-9s-with-5 example | Works? | Best? |
|---|---|---|---|
| Large integer | `70905` | ✅ | ❌ complicated arithmetic |
| Float | `7.0905` | ✅ | ❌ precision issues |
| **Tuple** | `(7, 9, 5)` | ✅ | ✅ simple, clean |

---

### ✅ Why Tuples Win

No arithmetic needed — just use commas:
```python
hand_rank(fk_9s) → (7, 9, 5)
#                   ↑  ↑  ↑
#                   │  │  └── kicker (remaining card)
#                   │  └───── rank of the 4-of-a-kind (9s)
#                   └──────── hand type: Four of a Kind = 7
```

---

### 📚 Lexicographic Ordering

Python compares tuples (and strings) **left to right** — this is called **lexicographic ordering**:

```python
(7, 9, 5) > (7, 3, 2)
# 7 == 7 → tie, move to next
# 9 > 3  → (7, 9, 5) wins ✅

'help' > 'hello'
# h==h, e==e, l==l → move on
# p > l → 'help' wins ✅
```

> 💡 Same principle used for sorting words in a dictionary — that's why it's called **lexico**graphic (lex = words).


### 🃏 hand_rank() — Full Tuple Structure per Hand

Each hand is described as a tuple: `(hand_type, tiebreakers...)`

| # | Hand | Example | Tuple returned |
|---|---|---|---|
| 8 | Straight Flush | J high | `(8, 11)` |
| 7 | Four of a Kind | Four Aces + Q kicker | `(7, 14, 12)` |
| 6 | Full House | Eights over Kings | `(6, 8, 13)` |
| 5 | Flush | 10, 8 high | `(5, 10, 8, ...)` all 5 cards |
| 4 | Straight | J high | `(4, 11)` |
| 3 | Three of a Kind | Three 7s | `(3, 7, 5, 2)` |
| 2 | Two Pair | Jacks and 3s | `(2, 11, 3, kicker)` |
| 1 | One Pair | Pair of 2s, J high | `(1, 2, 11, ...)` all cards |
| 0 | High Card | Nothing | `(0, ranks...)` all cards high→low |

**Key rules:**
- The **first number** = hand type (0–8)
- The **remaining numbers** = tiebreakers in order of importance
- For ties within the same hand type, Python compares the tuple **left to right** (lexicographic order)
- Cards use numeric ranks: `Jack=11, Queen=12, King=13, Ace=14`


### 🔧 Implementing hand_rank() with Tuples

`hand_rank()` returns a tuple where:
- **Element 0** → hand type (0–8)
- **Remaining elements** → tiebreakers in order of importance

```python
# Straight flush examples:
# ranks 2-3-4-5-6  → (8, 6)   ← high card is 6
# ranks 6-7-8-9-T  → (8, 10)  ← high card is 10
# (8, 10) > (8, 6) → second hand wins ✅
```

---

### 🧩 The `kind(n, ranks)` Function

`kind(n, ranks)` checks if there are **n cards of the same rank** and returns that rank (or `None`/falsy if not found).

```python
kind(4, ranks)  # → 9  if hand has four 9s  (truthy ✅)
kind(4, ranks)  # → None if no four of a kind (falsy ❌)
```

**Dual purpose — used as both:**
- A **value** → returns the rank (e.g. `9` for four 9s)
- A **boolean test** → truthy if found, falsy if not

```python
# In hand_rank():
if kind(4, ranks):              # ← boolean test
    return (7, kind(4, ranks),  # ← value usage
               kind(1, ranks))
```

> ⚠️ This works because card ranks go from **2 to 14** — `0` is never a valid rank, so returning `0` or `None` as a "not found" value is safe and always falsy in Python.

> 📝 **Note:** Unlike Java, Python treats `0`, `None`, and empty values as falsy — this dual-return pattern is a clean Python idiom.


### ✅ Testing hand_rank() — Adding Assertions per Hand

```python
sf = "6C 7C 8C 9C TC".split()  # Straight Flush  → high card = T (10)
fk = "9D 9H 9S 9C 7D".split()  # Four of a Kind  → four 9s, kicker 7
fh = "TD TC TH 7C 7D".split()  # Full House       → three 10s, pair of 7s

assert hand_rank(sf) == (8, 10)    # straight flush, 10 high
assert hand_rank(fk) == (7, 9, 7)  # four of a kind: four 9s + 7 kicker
assert hand_rank(fh) == (6, 10, 7) # full house: three 10s + pair of 7s
```

**How to read each tuple:**

| Hand | Tuple | Explanation |
|---|---|---|
| `sf` (6C 7C 8C 9C TC) | `(8, 10)` | Straight flush = 8, high card = T = 10 |
| `fk` (9D 9H 9S 9C 7D) | `(7, 9, 7)` | Four of a kind = 7, four 9s, kicker = 7D |
| `fh` (TD TC TH 7C 7D) | `(6, 10, 7)` | Full house = 6, three 10s, pair of 7s |

> 💡 Notice `fk` returns `(7, 9, 7)` — the first `7` is the **hand type** (four of a kind), and the second `7` is the **kicker card** (7 of diamonds). Two different 7s meaning different things!


### 🔧 hand_rank() — Complete Solution (All 9 Hand Types)

**Exercise:** fill in the return tuples for hands 6 → 0 using `kind()`, `flush()`, `straight()`, `two_pair()`, and `card_ranks()`.

```python
def hand_rank(hand):
    ranks = card_ranks(hand)
    if straight(ranks) and flush(hand):              # 8 — straight flush
        return (8, max(ranks))
    elif kind(4, ranks):                             # 7 — four of a kind
        return (7, kind(4, ranks), kind(1, ranks))
    elif kind(3, ranks) and kind(2, ranks):          # 6 — full house
        return (6, kind(3, ranks), kind(2, ranks))
    elif flush(hand):                                # 5 — flush
        return (5, ranks)
    elif straight(ranks):                            # 4 — straight
        return (4, max(ranks))
    elif kind(3, ranks):                             # 3 — three of a kind
        return (3, kind(3, ranks), ranks)
    elif two_pair(ranks):                            # 2 — two pair
        return (2, two_pair(ranks), ranks)
    elif kind(2, ranks):                             # 1 — one pair
        return (1, kind(2, ranks), ranks)
    else:                                            # 0 — high card
        return (0, ranks)
```

**Design decisions explained:**

| Hand | Tuple structure | Tiebreakers |
|---|---|---|
| Straight flush (8) | `(8, max(ranks))` | highest card |
| Four of a kind (7) | `(7, quad_rank, kicker)` | rank of the 4, then kicker |
| Full house (6) | `(6, triple_rank, pair_rank)` | rank of 3-of-a-kind first, then pair |
| Flush (5) | `(5, ranks)` | all 5 ranks in descending order |
| Straight (4) | `(4, max(ranks))` | highest card |
| Three of a kind (3) | `(3, triple_rank, ranks)` | rank of triple + remaining descending |
| Two pair (2) | `(2, (high_pair, low_pair), ranks)` | both pairs + kicker |
| One pair (1) | `(1, pair_rank, ranks)` | pair rank + remaining descending |
| High card (0) | `(0, ranks)` | all 5 ranks descending |

> 💡 `card_ranks()` returns ranks **sorted highest → lowest**, so when there is no special hand, just returning the full ranks list is enough — the highest card is automatically the first tiebreaker.

> 💡 For flush and high card, the whole `ranks` tuple is the tiebreaker — Python compares tuples element by element, so this correctly handles all edge cases automatically.


### 🃏 Helper Function: card_ranks()

**Purpose:** convert a hand (list of card strings) into a list of ranks **sorted highest → lowest**.

> 💡 Always sorted descending because ranks are used as tiebreakers — the highest card must come first.

**New test assertions added to `test()`:**

```python
assert card_ranks(sf) == [10, 9, 8, 7, 6]   # straight flush: T=10, descending
assert card_ranks(fk) == [9, 9, 9, 9, 7]    # four 9s first, then kicker 7
assert card_ranks(fh) == [10, 10, 10, 7, 7]  # three 10s first, then pair of 7s
```

| Hand | Cards | card_ranks output | Why? |
|---|---|---|---|
| `sf` | 6C 7C 8C 9C TC | `[10, 9, 8, 7, 6]` | T=10, sorted high→low |
| `fk` | 9D 9H 9S 9C 7D | `[9, 9, 9, 9, 7]` | four 9s before the 7 kicker |
| `fh` | TD TC TH 7C 7D | `[10, 10, 10, 7, 7]` | three 10s before the pair of 7s |

> 💡 `card_ranks()` only cares about **rank**, not suit. Face cards map to integers: J=11, Q=12, K=13, A=14, T=10.


### 🔧 Implementing card_ranks() — Mapping Face Cards to Integers

**Problem:** pulling the rank character directly gives wrong ordering — `'T'` sorts alphabetically *after* `'A'`, `'Q'`, etc., which is incorrect.

**Solution:** use `.index()` on a reference string where position = value:

```python
def card_ranks(cards):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in cards]
    ranks.sort(reverse=True)
    return ranks

# card_ranks(['AC', '3D', '4S', 'KH']) => [14, 13, 4, 3]
```

**How the index trick works:**

```
'--23456789TJQKA'
  0123456789...
         ↑ T is at index 10 → maps to 10
              ↑ J = 11, Q = 12, K = 13, A = 14
```

| Character | Index in string | Numeric rank |
|---|---|---|
| `'2'` | 2 | 2 |
| `'9'` | 9 | 9 |
| `'T'` | 10 | 10 |
| `'J'` | 11 | 11 |
| `'Q'` | 12 | 12 |
| `'K'` | 13 | 13 |
| `'A'` | 14 | 14 |

> 💡 The `--` at position 0 and 1 are placeholder dashes — no card has rank 0 or 1. This keeps the indexing aligned so `'2'` lands at index 2 naturally.

> 💡 Alternative approaches (dict lookup, if-elif chain) would work too, but the index string is the most concise.


### 🔀 Helper Functions: straight() and flush()

**Tests added first:**

```python
assert straight([9, 8, 7, 6, 5]) == True   # consecutive ranks → straight
assert straight([9, 8, 8, 6, 5]) == False  # pair: not a straight
assert flush(sf) == True                    # sf = all clubs → flush
assert flush(fk) == False                   # fk = 4 different suits → not a flush
```

**Solutions:**

```python
def straight(ranks):
    "Return True if the ordered ranks form a 5-card straight."
    return (max(ranks) - min(ranks) == 4) and (len(set(ranks)) == 5)

def flush(hand):
    "Return True if all the cards have the same suit."
    suits = [s for r, s in hand]
    return len(set(suits)) == 1
```

**straight() logic:**

| Condition | Why? |
|---|---|
| `max - min == 4` | 5 consecutive cards span exactly 4 (e.g. 5→9: 9−5=4) |
| `len(set(ranks)) == 5` | all 5 ranks are different (rules out pairs like [9,8,8,6,5]) |

> 💡 Both conditions together are enough — 5 distinct ranks spanning 4 = guaranteed consecutive sequence.

**flush() logic:**

- Extract suits: `[s for r, s in hand]` — each card is a 2-char string, `s` is the second character
- `set(suits)` collapses duplicates → if only 1 unique suit remains, all cards share the same suit


### 🔧 Helper Function: kind(n, ranks)

**Goal:** return the first rank that appears exactly `n` times in the hand. Return `None` if not found.

```python
def kind(n, ranks):
    """Return the first rank that this hand has exactly n of.
    Return None if there is no n-of-a-kind in the hand."""
    for r in ranks:
        if ranks.count(r) == n:
            return r
    return None
```

**How it works:**
- Loop through each rank in the (sorted) list
- `.count(r)` counts how many times `r` appears in the list
- If it matches exactly `n` → return that rank
- If nothing matches → return `None` (falsy)

---

**Test assertions:**

```python
tp = "5S 5D 9H 9C 6S".split()  # Two Pair: pair of 9s and pair of 5s, kicker 6
fkranks = card_ranks(fk)        # [9, 9, 9, 9, 7]
tpranks = card_ranks(tp)        # [9, 9, 6, 5, 5]

assert kind(4, fkranks) == 9    # four 9s → returns 9
assert kind(3, fkranks) == None # no three of a kind (it's four!) → None
assert kind(2, fkranks) == None # no pair → None
assert kind(1, fkranks) == 7    # one 7 (the kicker) → returns 7
```

**Key insight — exactly n, not at least n:**

| Call | fkranks = [9,9,9,9,7] | Why? |
|---|---|---|
| `kind(4, fkranks)` | `9` | four 9s → exactly 4 ✅ |
| `kind(3, fkranks)` | `None` | 9 appears 4 times, not 3 ❌ |
| `kind(2, fkranks)` | `None` | no rank appears exactly 2 times ❌ |
| `kind(1, fkranks)` | `7` | 7 appears exactly once ✅ |

> 💡 `kind()` uses `.count()` — a built-in list method that counts occurrences of an element.
> Since ranks are sorted **highest → lowest**, it always returns the **highest** matching rank first.


### 🔧 Helper Function: two_pair(ranks)

**Goal:** if there are two pairs, return `(highest_pair, lowest_pair)`; otherwise `None`.

```python
def two_pair(ranks):
    """If there are two pair, return the two ranks as a
    tuple: (highest, lowest); otherwise return None."""
    pair  = kind(2, ranks)           # highest pair (ranks sorted high→low)
    lowpair = kind(2, ranks[::-1])   # lowest pair (scan from the end)
    if pair and lowpair and pair != lowpair:
        return (pair, lowpair)
    return None
```

**How it works:**

| Step | Code | Result for `tpranks = [10, 10, 9, 7, 7]`* |
|---|---|---|
| Find highest pair | `kind(2, ranks)` | `10` — first pair found left→right |
| Find lowest pair | `kind(2, ranks[::-1])` | `7` — first pair found right→left |
| Check they differ | `pair != lowpair` | `10 != 7` ✅ → return `(10, 7)` |
| Single pair case | `pair == lowpair` | same rank = only 1 pair → `None` |

> 💡 `ranks[::-1]` reverses the list — since ranks are sorted **high → low**, reversing gives **low → high**, so `kind()` finds the lowest pair first.

> 💡 The check `pair != lowpair` is necessary to avoid returning `(9, 9)` when there's a three-of-a-kind — `kind(2, ...)` would find the same rank from both ends.

---

### ✅ Lesson 1 — Complete Program Summary

All helper functions implemented and tested:

```python
def card_ranks(hand):   # rank chars → sorted ints [14..2]
def straight(ranks):    # max-min==4 and 5 unique ranks
def flush(hand):        # all suits identical
def kind(n, ranks):     # first rank appearing exactly n times
def two_pair(ranks):    # (high_pair, low_pair) or None
def hand_rank(hand):    # returns comparable tuple (0–8, tiebreakers...)
def poker(hands):       # max(hands, key=hand_rank)
```

> 💡 Norvig closes Lesson 1 noting the test suite is not exhaustive yet — in the next videos he'll show an edge case that breaks the program and introduce more rigorous testing strategies.


---

## 🐛 Edge Case: Ace-Low Straight (A-2-3-4-5)

The Ace can count as **low** only in `A-2-3-4-5` (the "wheel"). Our program breaks this in 2 ways:
- `ranks = [14,5,4,3,2]` → max-min = 12 ≠ 4 → **not detected as straight**
- Even if detected, `max(ranks) = 14` → **wrong high card** (should be 5)

**Naïve fix:** change `card_ranks()`, `straight()`, and `hand_rank()`.

### 💡 Norvig's Design Principle
> *"The amount of change in code should be proportional to the amount of change in the concept."*

One conceptual change → one code change. Fix it **only in `card_ranks()`**: replace `[14,5,4,3,2]` with `[5,4,3,2,1]`. The other functions need zero changes.


### ✅ Fix: Ace-Low Straight in card_ranks()

Only `card_ranks()` needs to change — one special-case check at the end:

```python
def card_ranks(hand):
    "Return a list of the ranks, sorted with higher first."
    ranks = ['--23456789TJQKA'.index(r) for r, s in hand]
    ranks.sort(reverse=True)
    return [5, 4, 3, 2, 1] if ranks == [14, 5, 4, 3, 2] else ranks
```

**Why this works:**
- `[14, 5, 4, 3, 2]` is the only case where Ace plays low
- Replaced with `[5, 4, 3, 2, 1]` → `max-min = 5-1 = 4` ✅ and 5 unique values ✅
- `straight()` and `hand_rank()` need **zero changes**

```python
assert straight(card_ranks("AC 2D 4H 3D 5S".split())) == True  ✅
```

> 💡 `1` is only ever returned in this single edge case — all other ranks are 2–14.


### 🔁 Handling ties with `allmax`

Define a helper that returns all items equal to the maximum (optionally using a key).

```python
def allmax(iterable, key=None):
    result = []
    best = None
    for x in iterable:
        k = x if key is None else key(x)
        if not result or k > best:
            result = [x]
            best = k
        elif k == best:
            result.append(x)
    return result

def poker(hands):
    """Return list of winning hand(s)."""
    return allmax(hands, key=hand_rank)
```

Tests below ensure ties are handled (see next cell).

### 🎴 Dealing Cards – the `deal` function

You need a helper that takes:

```python
def deal(numhands, n=5, deck=mydeck):
    """Return a list of `numhands` poker hands with `n` cards each.
    `deck` is a list of card strings (default 52-card deck).
    """
```

**What to do:**
1. **Make a copy of the deck** so the caller's deck isn't mutated:
   `d = deck[:]`
2. **Shuffle** the copy using `random.shuffle(d)` (in-place modification).
3. **Slice out** `n` cards for each hand. With hands numbered 0..numhands‑1,
   the `i`th hand is `d[i*n:(i+1)*n]`.
4. **Return** the list of all hands.

```python
import random

mydeck = [r+s for r in '23456789TJQKA' for s in 'SHDC']

def deal(numhands, n=5, deck=mydeck):
    d = deck[:]                # copy
    random.shuffle(d)          # random order
    return [d[i*n:(i+1)*n] for i in range(numhands)]
```

> 💡 using a default deck lets you call `deal(4)` to get four 5‑card hands.
> You can also pass a custom deck or change `n` to deal 7‑card hands.

The quiz text hints at `random.shuffle`; the list comprehension forms the deck itself (all combinations of ranks and suits).

**Example usage:**

```python
>>> deal(2)               # two 5-card hands
[['7H', 'KD', '2C', 'AS', '9D'],
 ['5S', '3H', 'TC', 'AH', '4D']]

>>> deal(3, n=7)          # three 7-card hands
[['QC','2S','9H','8D','5C','JD','6S'],
 ['3C','KH','4S','6D','TS','AD','7H'],
 ['2D','JC','8S','KD','9C','4H','QS']]
```

Each sublist is one hand; the order is random so you get different cards every time. You can see the slicing: the first 5 (or 7) cards go to player 1, the next 5 (or 7) to player 2, etc.

## 📊 Poker Hand Probabilities – Validating Our Code

### The Wikipedia Reference Table

Wikipedia lists the probability of each poker hand type out of all possible 5-card hands:

| Rank | Hand | Approximate % |
|------|------|--------------|
| 1 (best) | Straight Flush | ~0.0015% |
| 2 | Four of a Kind | ~0.024% |
| 3 | Full House | ~0.14% |
| 4 | Flush | ~0.20% |
| 5 | Straight | ~0.39% |
| 6 | Three of a Kind | ~2.11% |
| 7 | Two Pair | ~4.75% |
| 8 | One Pair | ~42.26% |
| 9 (worst) | High Card | ~50.12% |

> Key insight: **rarer hands rank higher** — poker hand rankings are perfectly ordered by probability. The game is mathematically sound.

---

### How Many Hands Do We Need to Sample?

To validate our `hand_rank` function against these probabilities we need to simulate many random deals. But how many?

| Option | Verdict |
|--------|---------|
| ~52 (one per card) | ❌ Far too few — number of cards is irrelevant |
| ~1,000 per card (~52,000) | ❌ Still not enough for the rarest hands |
| **~700,000** | ✅ Target answer |
| 52! (all permutations) | ❌ Astronomically large, unnecessary |

**The reasoning:**  
We want **at least ~10 occurrences of the least common hand** (straight flush, ≈ 1 in 65,000) so that random variance doesn't skew our estimate badly.

$$\text{needed samples} \approx \frac{10}{P(\text{straight flush})} \approx \frac{10}{0.0000154} \approx 700{,}000$$

If we only expected 1 straight flush, sampling noise would make results meaningless (0 or 2 are equally likely). With ~10, we're reliably in the right ballpark.

---

### `hand_percentages` – Simulating the Table

```python
import random

def hand_percentages(n=700_000):
    """
    Deal n random 5-card hands, count how many of each rank appear,
    and print the percentage for each hand type.
    Default n=700,000 (takes ~20-30 sec; use n=1000 for a quick test).
    """
    counts = [0] * 9          # one bucket per rank (index 0 = straight flush … 8 = high card)
    for _ in range(n // 10):  # deal 10 hands at a time for efficiency
        for hand in deal(10):
            counts[hand_rank(hand)[0]] += 1
    for i in range(9):
        print(f"{hand_names[i]:20s}: {100 * counts[i] / n:5.2f}%")
```

> ⚠️ Running with 700,000 deals takes 20-30 seconds. Use `n=1_000` for a quick check.

---

### Comparing Simulated vs. Exact Results

When run with a large `n`, the output looks something like:

```
straight_flush      :  0.00%   (Wikipedia: 0.0015%)
four_of_a_kind      :  0.02%   (Wikipedia: 0.024%)
full_house          :  0.14%   (Wikipedia: 0.144%)
flush               :  0.20%   (Wikipedia: 0.197%)
straight            :  0.38%   (Wikipedia: 0.392%)
three_of_a_kind     :  2.09%   (Wikipedia: 2.113%)
two_pair            :  4.72%   (Wikipedia: 4.754%)
pair                : 42.30%   (Wikipedia: 42.257%)
high_card           : 50.15%   (Wikipedia: 50.118%)
```

The simulated percentages closely match the exact mathematical values — confirming our `hand_rank` logic is correct. This is a form of **empirical verification via Monte Carlo simulation**.


## 🧭 Dimensions of Programming

A program lives in a **multidimensional space** defined by four key axes. Every design decision moves it along one or more of these dimensions:

| Axis | Question |
|------|----------|
| ✅ **Correctness** | Does it do the right thing? |
| ⚡ **Efficiency** | Is it fast enough? |
| 🧩 **Features** | What exactly does it do? |
| 💎 **Elegance** | Is it clear, simple, and general? |

```
        Correctness
             ▲
             │         ★ ideal program
             │        /
             │       /
─────────────┼──────────────► Features
             │      /
             │     /
         (Efficiency & Elegance are the other axes — think 4D space)
```

### 💎 Elegance is not optional
> *"Elegance is not optional."* — Richard O'Keefe

Elegance is made up of **"-ity" qualities**:
- **Clarity** — easy to read and understand
- **Simplicity** — no unnecessary complexity
- **Generality** — works beyond just the current case

Improving elegance doesn't add features today, but it **pays off in the future**: a more elegant program is cheaper to maintain, extend, and debug.

### ⚖️ Voltaire's Rule
> *"The best is the enemy of the good."*

Don't chase 100% perfection in one dimension at the cost of all others. Good engineering means making **smart tradeoffs** — knowing when to stop and move on.


## ♻️ Refactoring – Moving Along the Elegance Axis

**Refactoring** = changing a program to be cleaner/clearer *without* changing what it does.

The original `hand_rank` repeated itself checking for pairs and triples (`3 of a kind AND 2 of a kind` → full house). That violates the **DRY principle**:

> *"Don't Repeat Yourself"* — every piece of knowledge should appear once and only once.

---

### The `group` Helper

Instead of asking "is there a pair? is there a triple?" separately, we create a richer representation upfront:

```python
def group(hand):
    """Return (counts, ranks) sorted by count desc, then rank desc."""
    ranks = [r for r, s in hand]
    return (sorted([ranks.count(r) for r in set(ranks)], reverse=True),
            sorted(set(ranks), key=ranks.count, reverse=True))
```

**Example** — hand `[7, 10, 7, 9, 7]`:

| | Value |
|---|---|
| `counts` | `[3, 1, 1]` — three 7s, one 10, one 9 |
| `ranks`  | `[7, 10, 9]` — ordered by count |

---

### Refactored `hand_rank`

```python
def hand_rank(hand):
    groups = group(hand)
    counts, ranks = groups
    if ranks == (14, 5, 4, 3, 2):   # ace-low straight
        ranks = (5, 4, 3, 2, 1)
    straight = len(set(ranks)) == 5 and max(ranks) - min(ranks) == 4
    flush    = len(set(s for r, s in hand)) == 1
    return (9 if counts == [5]          else   # five of a kind
            8 if straight and flush     else   # straight flush
            7 if counts == [4, 1]       else   # four of a kind
            6 if counts == [3, 2]       else   # full house
            5 if flush                  else   # flush
            4 if straight               else   # straight
            3 if counts == [3,1,1]      else   # three of a kind
            2 if counts == [2,2,1]      else   # two pair
            1 if counts == [2,1,1,1]    else   # one pair
            0), ranks                          # high card
```

No repetition — each hand type is checked **once** using its count signature.

---

### 🤯 Bonus Insight – Poker Hands as Partitions of 5

The count patterns are exactly the **integer partitions of 5**:

| Partition | Hand |
|-----------|------|
| `[5]` | Five of a kind |
| `[4,1]` | Four of a kind |
| `[3,2]` | Full house |
| `[3,1,1]` | Three of a kind |
| `[2,2,1]` | Two pair |
| `[2,1,1,1]` | One pair |
| `[1,1,1,1,1]` | High card |

All **7 integer partitions of 5**, in **lexicographic order** — which is also exactly the **poker ranking order** from best to worst. Pure math, built into the game.


## 🔧 `group`, `unzip`, and the Lookup Table Approach

### `group` and `unzip`

```python
def group(items):
    """Return a sorted list of (count, item) pairs, highest count first."""
    return sorted([(items.count(x), x) for x in set(items)], reverse=True)

def unzip(pairs):
    """Convert a list of (count, item) pairs into (counts, items)."""
    return zip(*pairs)
```

- `group` counts occurrences of each item and sorts biggest-first.
- `unzip` (via `zip(*pairs)`) converts a list of pairs into a pair of lists — a standard Python trick.

---

### Lookup Table – One More Refactor

Instead of a long `if/elif` chain, store the partition → ranking mapping in a dict:

```python
count_rankings = {(5,): 10, (4,1): 7, (3,2): 6,
                  (3,1,1): 3, (2,2,1): 2, (2,1,1,1): 1, (1,1,1,1,1): 0}

def hand_rank(hand):
    counts, ranks = unzip(group(hand))
    if ranks == (14,5,4,3,2): ranks = (5,4,3,2,1)
    straight = len(set(ranks)) == 5 and max(ranks) - min(ranks) == 4
    flush    = len(set(s for r,s in hand)) == 1
    return max(count_rankings[counts], 4*straight + 5*flush), ranks
```

The key trick: `4*straight + 5*flush` exploits bool→int conversion:

| straight | flush | expression | result |
|----------|-------|-----------|--------|
| False | False | 0+0 | 0 |
| True | False | 4+0 | 4 ✅ straight |
| False | True | 0+5 | 5 ✅ flush |
| True | True | 4+5 | 9 ✅ straight flush |

9 lines of `if/elif` collapsed to **1 line** — more concise, but less explicit. Both are valid; it's a matter of taste.

---

## 📝 Lessons Learned

| Step | Principle |
|------|-----------|
| 1 | **Understand the problem** — read the spec, ask questions |
| 2 | **Define the pieces** — cards, hands, ranks, suits… |
| 3 | **Reuse what exists** — `max()`, `random.shuffle()`, `zip()` |
| 4 | **Write tests** — you don't know what your program does without them |
| 5 | **Explore the design space** — correctness → efficiency → features → elegance |
| 6 | **Use good taste to know when to stop** — don't over-engineer |


---

# 🃏 Bonus: Shuffling

## The Teacher's Shuffle vs. Knuth's Algorithm P

### Teacher's shuffle (❌ wrong)
```python
# swap random pairs until every position has been swapped at least once
swapped = [False] * N
while not all(swapped):
    i, j = random_indices()
    swap(deck[i], deck[j])
    swapped[i] = swapped[j] = True
```
**Problems:**
- **Not guaranteed to terminate** — the while-loop could theoretically run forever.
- **O(N²)** — the last unswapped element takes ~N tries to hit, so total ≈ N×N swaps.
- **Biased** — not all permutations are equally probable.

---

### Knuth's Algorithm P (✅ correct)
```python
def shuffle(deck):
    N = len(deck)
    for i in range(N - 1):
        j = random.randrange(i, N)   # pick from i..N-1
        deck[i], deck[j] = deck[j], deck[i]
```
- **O(N)** — exactly one swap per element.
- **Fair** — every permutation is equally probable.
- Using `N` instead of `N-1` in the range would be fine (still fair) but wastes one no-op swap at the end.

---

## Comparing Shuffles

| Algorithm | Complexity | Fair? |
|-----------|-----------|-------|
| `random.shuffle` (Knuth) | O(N) | ✅ |
| Teacher's `shuffle1` | O(N²) | ❌ biased |
| `shuffle2` (partial fix) | O(N²) | ✅ correct but slow |
| `shuffle3` (swap each with any) | O(N) | ❌ biased |

> **Use `random.shuffle`.** O(N) and provably fair.

---

## Pure Functions vs. Subroutines

| | Pure function | Subroutine (impure) |
|---|---|---|
| Example | `sorted(lst)` | `lst.sort()` |
| Returns | new value | `None` |
| Modifies input? | ❌ | ✅ |
| Test in 1 line? | ✅ `assert sorted([3,2,1]) == [1,2,3]` | ❌ need setup + call + inspect |

**Prefer pure functions** when possible — they are easier to test and reason about. Use subroutines only when managing significant shared state.


---

# 🐍 Andy's Corner 1 — List Comprehensions

## What is a List Comprehension?

A concise, Pythonic way to build a list in **one line** — faster and more readable than a for loop with `.append()`.

```python
# ❌ for-loop approach (verbose)
result = []
for x in iterable:
    result.append(f(x))

# ✅ list comprehension (Pythonic)
result = [f(x) for x in iterable]
```

---

## With Tuples (unpacking)

When each item in the iterable is a tuple, unpack all fields:

```python
ta_data = [('Peter', 'USA', 'CS212'),
           ('Sarah', 'France', 'CS101'),
           ('Gundega', 'Latvia', 'CS387')]

# ✅ readable unpacking
ta_facts = [name + " lives in " + country + " and TAs " + course
            for name, country, course in ta_data]
```

> Always unpack **all** fields in the tuple — leaving one out causes a `ValueError: too many values to unpack`.

---

## Filtering with `if`

Add an `if` clause to include only items that match a condition:

```python
# Only TAs outside the USA
remote = [name + " lives in " + country + " and TAs " + course
          for name, country, course in ta_data
          if country != 'USA']
```

---

## Pattern Summary

```python
[expression  for var in iterable]               # basic
[expression  for a, b, c in iterable]           # tuple unpacking
[expression  for a, b, c in iterable  if cond]  # with filter
```

**Exercise:** find all TAs teaching a 300-level course:
```python
ta_300 = [name for name, country, course in ta_data if course.startswith('3')]
# ['Gundega', 'Job']
```


### ✅ Exercise Solution — 300-level TAs

```python
ta_data = [['Peter',   'USA',     'CS262'],
           ['Andy',    'USA',     'CS212'],
           ['Sarah',   'England', 'CS101'],
           ['Gundega', 'Latvia',  'CS373'],
           ['Job',     'USA',     'CS387'],
           ['Sean',    'USA',     'CS253']]

ta_300 = [name + " is the TA for " + course
          for name, country, course in ta_data
          if course.find('CS3') != -1]

# → ['Gundega is the TA for CS373', 'Job is the TA for CS387']
```

**Key points:**
- `country` must be included in the unpacking even though it's not used — each entry has 3 elements and all must be referenced.
- `course.find('CS3') != -1` — `find()` returns `-1` when the substring is **not** found; `!= -1` means "CS3 was found" → 300-level course.
- `CS373` and `CS387` both contain `'CS3'` ✅ — `CS253` does not ❌.


---

# 🃏 Poker Rules — Quick Reference

Only one thing matters here: how 5-card hands are ranked (numbered 0–8, Python style):

| # | Hand | What it is |
|---|------|-----------|
| 8 | Straight Flush | 5 consecutive, same suit — `9H TH JH QH KH` |
| 7 | Four of a Kind | 4 same rank — `AS AH AD AC 6D` |
| 6 | Full House | 3 of one + 2 of another — `TH TC TS 2H 2D` |
| 5 | Flush | 5 same suit — `3S 6S JS QS KS` |
| 4 | Straight | 5 consecutive, any suit — `5S 6D 7C 8D 9S` |
| 3 | Three of a Kind | 3 same rank — `4H 4D 4C QS 2H` |
| 2 | Two Pair | 2 pairs — `JH JD 9H 9C 3S` |
| 1 | Pair | 2 same rank — `3H 3D 8C 5H 2S` |
| 0 | High Card | Nothing — `KH 9D 7S 4C 2H` |

Cards: `rank + suit` (e.g. `TH` = Ten of Hearts). Ties broken by card values.


---

## 🂡 HW1 — 7-Card Stud: `best_hand`

In 7-card stud you're dealt 7 cards but only play the best 5. The task: find that best 5-card hand.

**Key insight:** generate all possible 5-card combinations from 7 cards and pick the one with the highest `hand_rank`.

```python
import itertools

def best_hand(hand):
    "From a 7-card hand, return the best 5 card hand."
    return max(itertools.combinations(hand, 5), key=hand_rank)
```

**How it works — 3 steps:**

1. **`combinations(hand, 5)`** → generates all 21 possible 5-card subsets from the 7 cards
2. **`hand_rank`** → scores each combination with a comparable tuple e.g. `(8, 10)` for a straight flush
3. **`max`** → returns the combination that produced the highest tuple

```
7 cards → 21 combos → hand_rank each one → max → best 5-card hand
```

> Works for any number of input cards — not just 7.


---

## 🃏 HW1-2 — Jokers Wild: `best_wild_hand`

Two jokers act as **wild cards**, restricted by color:

| Joker | Can replace |
|-------|------------|
| `?B` (black) | Any spade `S` or club `C` |
| `?R` (red)   | Any heart `H` or diamond `D` |

**Strategy:**
1. For each card in the hand, define its possible values — normal cards map to themselves, jokers map to all 26 cards of their color.
2. Use `itertools.product` to generate every combination of substitutions.
3. For each substituted hand, find the best 5-card selection with `best_hand`.
4. Return the overall best.

```python
allranks   = '23456789TJQKA'
black_cards = [r+s for r in allranks for s in 'SC']  # 26 black cards
red_cards   = [r+s for r in allranks for s in 'HD']  # 26 red cards

def replacements(card):
    if card == '?B': return black_cards
    if card == '?R': return red_cards
    return [card]   # normal card → only itself

def best_wild_hand(hand):
    "Try all values for jokers in all 5-card selections."
    hands = set(best_hand(list(h))
                for h in itertools.product(*map(replacements, hand)))
    return max(hands, key=hand_rank)
```

**How it works:**
- `map(replacements, hand)` → for each card returns its list of options (1 option for normal cards, 26 for jokers)
- `itertools.product(...)` → generates every possible substituted hand (up to 26² = 676 if two jokers)
- `best_hand` → picks best 5 from each substituted hand
- `max(..., key=hand_rank)` → returns the overall best

> `set(...)` avoids evaluating duplicate hands when the same card appears multiple times.


---

# 🧩 Lesson 2 — Zebra Puzzle

## 📋 The Puzzle

**15 clues. 5 houses. Find: who drinks water? who owns the zebra?**

Each house is a different color; inhabitants differ in nationality, pet, drink, and cigarette brand.

| # | Clue |
|---|------|
| 1 | There are five houses |
| 2 | The Englishman lives in the red house |
| 3 | The Spaniard owns the dog |
| 4 | Coffee is drunk in the green house |
| 5 | The Ukrainian drinks tea |
| 6 | The green house is immediately to the right of the ivory house |
| 7 | The Old Gold smoker owns snails |
| 8 | Kools are smoked in the yellow house |
| 9 | Milk is drunk in the middle house |
| 10 | The Norwegian lives in the first house |
| 11 | The Chesterfields smoker lives next to the man with the fox |
| 12 | Kools are smoked next to the house with the horse |
| 13 | The Lucky Strike smoker drinks orange juice |
| 14 | The Japanese smokes Parliaments |
| 15 | The Norwegian lives next to the blue house |


## 📦 Concepts Inventory

| Concept | Description |
|---|---|
| **Houses** | 5 numbered positions (1–5) |
| **Properties** | Nationality, color, pet, drink, smoke |
| **Property group** | 5 mutually exclusive values assigned to the 5 houses |
| **Assignment** | Match each property value to exactly one house |
| **Location relations** | first, middle, next-to, immediately-right-of |

**Key design decision — do we need to name property groups?**

Yes. Properties **within a group are mutually exclusive** (if red → house 2, no other color can → house 2), but properties from **different groups are not** (orange juice can also → house 2). The program must track this.

Two valid approaches:
- Named groups: `nationality = [Englishman, Spaniard, ...]`
- Unnamed groups: manage grouping implicitly in the iteration

Ignoring groups entirely **does not work**.


## 🔧 Representing Assignments

Three valid implementation styles — all work:

```python
# Option A — house as set of properties
houses[1].add('red')

# Option B — house as object with fields
house1.color = 'red'

# Option C — property as variable (simplest ✅)
red = 1   # house number assigned to the property
```

**Norvig's choice: Option C** — assign a house number to each property variable. Simplest, and simple is enough until proven otherwise.


## 📐 Back-of-Envelope: Brute Force Feasibility

### How many combinations?

- Each property group has **5! = 120** possible orderings (permutations of 5 values across 5 houses)
- There are **5 property groups** → total = $5!^5$

$$5!^5 = 120^5 \approx 24{.}9 \text{ billion}$$

**Quick estimate:** round 120 → 100, then $100^5 = 10^{10}$ (10 billion). Round back up → ~20 billion. ✅

### Is it feasible?

| Scale | Verdict |
|-------|---------|
| Millions | ✅ Happy valley — fast |
| **Billions** | ❓ Uncertain — need to refine |
| Trillions | ❌ Infeasible — need a better algorithm |

At ~1 billion instructions/sec, **24.9 billion iterations ≈ ~1 hour** for a naïve brute force. Not fast enough — but can we prune?


## 🏗️ Brute Force Structure

```python
import itertools

houses = (1, 2, 3, 4, 5)
orderings = list(itertools.permutations(houses))  # 120 orderings

for (red, green, ivory, yellow, blue) in orderings:          # colors
  for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings:  # nationalities
    for (dog, snails, fox, horse, zebra) in orderings:       # pets
      for (OJ, tea, coffee, milk, water) in orderings:       # drinks
        for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings:  # smokes
          if <all constraints satisfied>:
              solution found
```

- 5 nested loops × 120 orderings each = $120^5 \approx 25$ billion iterations
- Each property variable gets its house number directly (Option C representation)
- Constraints are checked **after** all 5 loops are nested — no early pruning

> 💡 `itertools.permutations(houses)` generates all 120 orderings of `(1,2,3,4,5)` — one per possible assignment of a property group to the 5 houses.

**Estimated runtime: ~1 hour** — in the uncertain zone. Can we improve by checking constraints earlier?


## ⏱️ Runtime Estimate (Back of Envelope)

$$5!^5 = 120^5 \approx 24.9 \text{ billion iterations}$$

| Instructions per iteration | Time at 2.4 GHz |
|---|---|
| 1 (impossible) | ~10 sec |
| 100 | ~16 min |
| **1,000 (realistic)** | **~160 min ≈ 2–3 hours** |

**Conclusion:** brute force with all constraints checked at the end → **~1–3 hours**. In the uncertain zone — but Norvig actually ran it to confirm. We need to prune constraints earlier to bring it into the "happy valley".


## 🔗 Encoding Constraints

With Option C (property = house number), each clue becomes a simple equality or helper call:

```python
# Clue 2: The Englishman lives in the red house
Englishman == red

# Clue 3: The Spaniard owns the dog
Spaniard == dog

# Clue 6: The green house is immediately to the right of the ivory house
imright(green, ivory)

# Clue 11: Chesterfields smoker lives next to the man with the fox
nextto(Chesterfields, fox)
```

### Helper functions

```python
def imright(h1, h2):
    "House h1 is immediately right of h2 if h1-h2 == 1."
    return h1 - h2 == 1

def nextto(h1, h2):
    "Two houses are next to each other if they differ by 1."
    return abs(h1 - h2) == 1
    # equivalently: imright(h1, h2) or imright(h2, h1)
```


## 🧩 Complete Solution — `zebra_puzzle`

Uses a **generator expression** instead of nested for-loops — same logic, cleaner structure.

```python
import itertools

def zebra_puzzle():
    "Return (WATER, ZEBRA) — house numbers for who drinks water and who owns the zebra."
    houses = [1, 2, 3, 4, 5]
    orderings = list(itertools.permutations(houses))  # 120 orderings
    return next(
        (WATER, ZEBRA)
        for (red, green, ivory, yellow, blue)                      in orderings
        for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings
        for (dog, snails, fox, horse, zebra)                       in orderings
        for (OJ, tea, coffee, milk, WATER)                         in orderings
        for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings
        if Englishman == red                    # 2
        if Spaniard == dog                      # 3
        if coffee == green                      # 4
        if Ukrainian == tea                     # 5
        if imright(green, ivory)                # 6
        if OldGold == snails                    # 7
        if Kools == yellow                      # 8
        if milk == 3                            # 9 — middle house
        if Norwegian == 1                       # 10 — first house
        if nextto(Chesterfields, fox)           # 11
        if nextto(Kools, horse)                 # 12
        if LuckyStrike == OJ                    # 13
        if Japanese == Parliaments              # 14
        if nextto(Norwegian, blue)              # 15
    )
```

- `next(...)` returns the **first** yielded tuple from the generator — the solution.
- Each `if` is a **guard**: as soon as one fails, the inner loop skips that combination.
- All 15 constraints are encoded as simple equalities or helper calls.

> ⚠️ This version still checks all constraints **at the deepest nesting level** — runtime ~1–3 hours. See next section for the pruned version.


## 🔁 Generator Expressions vs. List Comprehensions

**List comprehension** → builds the entire list in memory:
```python
[s for r, s in cards]                            # all suits
[s for r, s in cards if r in ('J','Q','K')]      # only face card suits
```

**Generator expression** → same syntax but with `()` → produces values **on demand** (lazy):
```python
(s for r, s in cards)                            # generator — no list built yet
```

### How to read them — left to right (except the first term)

```
[ term  |  for a, b in iterable  |  if condition  |  for x in other  |  if ... ]
          ↑ read these first, left→right, as nested statements
                                                                         ↑ then evaluate term
```

The `term` at the front is evaluated **last**, once you reach the innermost level of nesting.

### `next()` + generator = get first match without computing all

```python
next(x for x in big_list if condition(x))   # stops at first match ✅
```

vs.

```python
[x for x in big_list if condition(x)][0]    # builds full list first ❌
```

> 💡 In `zebra_puzzle`, `next(...)` stops as soon as the first valid assignment is found — no wasted work after the solution is reached.


## ✂️ Pruning Zebra Puzzle — Move Constraints Up

**Problem:** all 15 constraints are checked at the deepest loop level → 25 billion combos.

**Fix:** move each constraint to the **earliest loop** where all its variables are already bound.

```python
return next(
    (WATER, ZEBRA)
    for (red, green, ivory, yellow, blue) in orderings
    if imright(green, ivory)                # ← moved up: only needs colors
    for (Englishman, Spaniard, Ukrainian, Norwegian, Japanese) in orderings
    if Englishman == red                    # ← moved up: needs colors + nationalities
    if Norwegian == 1
    if nextto(Norwegian, blue)
    for (dog, snails, fox, horse, ZEBRA) in orderings
    if Spaniard == dog
    for (OJ, tea, coffee, milk, WATER) in orderings
    if Ukrainian == tea
    if coffee == green
    if milk == 3
    for (OldGold, Kools, Chesterfields, LuckyStrike, Parliaments) in orderings
    if Kools == yellow
    if OldGold == snails
    if LuckyStrike == OJ
    if Japanese == Parliaments
    if nextto(Chesterfields, fox)
    if nextto(Kools, horse)
)
```

| Version | Time |
|---------|------|
| All constraints at end | ~1 hour |
| Constraints moved up | **< 1 ms** |

> 💡 Each early `if` prunes entire subtrees of combinations before they're explored. Same algorithm — drastically less work.


## ⏱️ `timedcall` — Timing Any Function

```python
import time

def timedcall(fn, *args):
    "Call fn(*args); return (elapsed_time, result)."
    t0 = time.perf_counter()
    result = fn(*args)
    t1 = time.perf_counter()
    return t1 - t0, result

# Usage
elapsed, answer = timedcall(zebra_puzzle)
# → (0.00028, (1, 5))
```

### `*args` — Pack & Unpack Arguments

| Location | Meaning |
|----------|---------|
| `def f(fn, *args)` | collect all extra args into a **tuple** called `args` |
| `fn(*args)` | **unpack** the tuple back into positional arguments |

```python
timedcall(max, [3,1,2])   # args = ([3,1,2],)  →  fn(*args) = max([3,1,2])
```

### Functions as First-Class Objects

Passing `fn` instead of `fn()` **delays execution** until the clock has started:

```python
timedcall(zebra_puzzle)   # ✅ passes the function — called inside, after t0
timedcall(zebra_puzzle()) # ❌ evaluates zebra_puzzle() first — clock misses it
```

> 💡 Functions are objects just like ints or lists — they can be passed, stored, and called later. This lets you control *when* execution happens.


## 🔁 `timedcalls` — Repeated Timing (Experimental Science)

One measurement isn't enough — repeat to reduce:
- **external events** (OS noise)
- **random variation**
- **measurement errors**

```python
def average(numbers):
    return sum(numbers) / float(len(numbers))

def timedcalls(n, fn, *args):
    """Call fn(*args) n times (int) or for up to n seconds (float).
    Returns (min, avg, max) of elapsed times."""
    if isinstance(n, int):
        times = [timedcall(fn, *args)[0] for _ in range(n)]
    else:
        times = []
        while sum(times) < n:
            times.append(timedcall(fn, *args)[0])
    return min(times), average(times), max(times)
```

| `n` type | Behavior |
|----------|----------|
| `int` (e.g. `10`) | repeat exactly 10 times |
| `float` (e.g. `10.0`) | repeat until 10 seconds have elapsed |

```python
timedcalls(5, zebra_puzzle)      # run 5 times
timedcalls(2.0, zebra_puzzle)    # run for ~2 seconds
# → (min, avg, max) in seconds
```

> 💡 `isinstance(n, int)` distinguishes `10` (int) from `10.0` (float) — Python treats them as different types even though the value looks the same.


## 🧩 Aspect-Oriented Programming

When designing a program, we juggle multiple concerns at once:

| Concern | What it addresses |
|---------|-------------------|
| **Correctness** | Does the program do what it should? |
| **Efficiency** | Does it run fast enough? |
| **Debugging** | Extra scaffolding to inspect/verify during dev |

The problem: all three get **interwoven** in the same code, creating a mess.

**Aspect-Oriented Programming (AOP)** is the idea of **separating these concerns** as much as possible — keep the core correctness logic distinct from efficiency and debugging code.

> 💡 We can't achieve perfect separation, but we should always strive to **minimize coupling** between these aspects.


## 🔢 Counting Iterations — Instrumenting with `c()`

When the solver is a **single generator expression**, you can't insert `count += 1` statements inside it. The solution: wrap `orderings` with a lightweight counting function `c()`.

```python
# Before: orderings used directly
for (red, green, ivory, yellow, blue) in orderings

# After: wrap with c() — minimal change, no logic altered
for (red, green, ivory, yellow, blue) in c(orderings)
```

`c()` is a **debugging tool** — short name is acceptable since it's temporary scaffolding.

### `instrument_fn` — Measure calls & items

```python
def instrument_fn(fn, *args):
    c.starts, c.items = 0, 0
    result = fn(*args)
    print(f'{c.starts} starts, {c.items} items')
    return result
```

| Metric | Meaning |
|--------|---------|
| `c.starts` | how many times the for loop started |
| `c.items` | how many items were yielded through |

> 💡 This gives you visibility into **how much work** the solver actually does, without changing the puzzle logic.


## ⚡ Generator Functions — `yield`

A **generator function** looks like a regular function but uses `yield` instead of `return`.

```python
def ints(start, end=None):
    i = start
    while i <= end or end is None:
        yield i
        i += 1
```

### How it works

- When called, it returns a **generator object** (not a list)
- Each call to `next()` runs until the next `yield`, returns that value, then **pauses**
- Resumes from the same point on the next `next()` call

```python
L = ints(0, 1_000_000)   # instant — no list created
next(L)   # → 0
next(L)   # → 1
# or: for i in L: ...
```

### Regular function vs Generator function

| | Regular function | Generator function |
|--|------------------|--------------------|
| Keyword | `return` | `yield` |
| Returns | a value (once) | a lazy sequence |
| Memory | stores all results | one item at a time |

### Infinite sequences

Pass `end=None` to get an **infinite generator**:

```python
ints(0)       # 0, 1, 2, 3, ... forever
```

The loop condition `while i <= end or end is None` keeps going when `end` is `None`.

> 💡 Generators are perfect for infinite or very large sequences — they only compute values **on demand**.


## 🔄 Exercise — `all_ints()`

Generate integers in the order: `0, +1, -1, +2, -2, +3, -3, ...`

```python
def all_ints():
    "Generate integers in the order 0, +1, -1, +2, -2, +3, -3, ..."
    yield 0
    for i in ints(1):        # infinite: 1, 2, 3, ...
        yield +i
        yield -i

# Alternative (explicit infinite loop):
def all_ints():
    yield 0
    i = 1
    while True:
        yield +i
        yield -i
        i += 1
```

> 💡 Both produce the same infinite sequence — `ints(1)` is more concise; `while True` is more explicit.


## 🔍 How `for` Loops Really Work

You might think `for x in items: print(x)` is just index-based iteration. The truth is deeper:

```python
# What Python actually does:
it = iter(items)          # 1. create an iterator
while True:
    try:
        x = next(it)      # 2. get next item
        print(x)          # 3. run the body
    except StopIteration:
        break             # 4. stop when exhausted
```

### Key concepts

| Term | Meaning |
|------|---------|
| **Iterable** | anything you can loop over: list, string, generator, ... |
| `iter(obj)` | creates an **iterator** from an iterable |
| `next(it)` | returns the next value; raises `StopIteration` when done |

- A **generator function** is already an iterator — `yield` handles `StopIteration` automatically
- Generators, lists, strings, and tuples all implement this same **iteration protocol**

> 💡 Because generators control *when* they produce values, we can count exactly how many items were consumed — this is how `c()` works.


## 🧮 `c()` — The Counting Generator

```python
def c(sequence):
    "Count the starts and items of a sequence."
    c.starts += 1
    for item in sequence:
        c.items += 1
        yield item
```

- `c.starts` increments each time the `for` loop begins a new `orderings` iteration
- `c.items` increments for every item yielded — i.e., items actually consumed
- Because `c` is a **generator**, it only counts items that are actually requested (early `if` filters stop it early)

```python
# Setup before calling the solver:
c.starts, c.items = 0, 0
instrument_fn(zebra_puzzle)
# → "25 starts, 2769 items"
```

> 💡 This separation is AOP in action: the counting concern lives in `c()`, not scattered across the puzzle logic.


## 📝 Zebra Puzzle — Lessons Learned

| Step | What we did |
|------|-------------|
| **Concept inventory** | Listed all entities and constraints before coding |
| **Simple first** | Implemented the straightforward (brute-force) solution |
| **Back-of-envelope estimate** | Calculated runtime before optimizing |
| **Refine** | Moved constraints up to prune the search space |
| **Build tools** | Built `timedcall`, `timedcalls`, `c()` as reusable debugging/profiling aids |
| **Separation of aspects** | Kept correctness, efficiency, and debugging code as separate as possible |

**Key takeaway:** By following a systematic process — concept inventory → simple implementation → estimate → refine — you can reliably arrive at an elegant, efficient solution without needing a clever flash of insight.

> 💡 The cleverness comes from the *process*, not luck.


---

## 🔐 Cryptarithmetic (Alphametics)

**Problem:** Each letter stands for a unique digit (0–9). Find the assignment that makes the arithmetic equation true.

**Example:**
```
ODD + ODD == EVEN
```

### Key observations (human inference)
- The leading digit of the result is the **carry digit** → must be `1` (e.g., `E = 1` in `ODD + ODD == EVEN`)
- `D + D = 2D` → result is always **even**, so `N` must be even

### Approach: Brute-force with permutations

Instead of encoding all arithmetic rules, just:
1. Extract all unique letters from the formula
2. Try all **permutations** of digits for those letters (`itertools.permutations`)
3. Substitute (translate) and `eval()` each filled-in formula
4. Return the first valid one

> ~10! ≈ 3 million permutations max, but usually far fewer (only as many letters as there are).

---

### Concepts & Representation

| Concept | Representation |
|---------|---------------|
| Original formula | `str` with uppercase letters |
| Filled-in formula | `str` with digits |
| Letter → digit mapping | `str.maketrans` + `str.translate` |
| Evaluation | `eval(f)` — evaluates a string as a Python expression |

### Key Python tools

```python
# Translation table: replaces A→1, B→2, C→3
table = str.maketrans('ABC', '123')
f = 'A + B == C'
f.translate(table)   # → '1 + 2 == 3'
eval('1 + 2 == 3')   # → True
```

---

### `valid(f)` — checks a filled-in formula

```python
import re

def valid(f):
    """True if f has no leading zeros and evals to True."""
    try:
        return not re.search(r'\b0[0-9]', f) and eval(f) is True
    except ArithmeticError:
        return False
```

- `\b0[0-9]` — regex that detects a **leading zero** in any number (e.g., `0405`)
  - Leading zeros are invalid AND cause octal interpretation bugs in Python 2
- Catches `ArithmeticError` (covers `ZeroDivisionError`, overflow, etc.)

---

### `fill_in(formula)` — generator of all substitutions

```python
import string, re, itertools

def fill_in(formula):
    letters = ''.join(set(re.findall('[A-Z]', formula)))  # unique uppercase letters
    for digits in itertools.permutations('0123456789', len(letters)):
        table = str.maketrans(letters, ''.join(digits))
        yield formula.translate(table)
```

- Uses `re.findall('[A-Z]', formula)` → all uppercase letters
- `set(...)` → remove duplicates; `''.join(...)` → back to string
- `itertools.permutations('0123456789', n)` → all ordered picks of `n` digits
- `yield` makes it a **generator** → lazy evaluation (stops as soon as a solution is found)

---

### `solve(formula)` — the main solver

```python
def solve(formula):
    """Return the first valid digit-filled version of formula, or None."""
    for f in fill_in(formula):
        if valid(f):
            return f
```

- Iterates `fill_in` lazily → stops at first valid result
- Clean separation: `solve` = control flow, `fill_in` = search space, `valid` = correctness check

---

### Design takeaways

- **Generator pattern** → separates *producing results* from *consuming them*; allows early exit
- **`eval` + string substitution** → avoids hand-coding arithmetic rules
- **3-function decomposition** (`solve` / `fill_in` / `valid`) keeps each piece simple and testable
- **`from __future__ import division`** (Python 2 only) — makes `/` behave like true division instead of integer division


---

## 🧪 Testing the Cryptarithmetic Solver

### Test harness pattern

```python
import time

examples = [
    'ODD + ODD == EVEN',
    'A**2 + B**2 == C**2',
    'AN + BN == CN and N > 1',
    # ...
]

start = time.clock()
for example in examples:
    print(example)
    t, result = timedcall(solve, example)
    print(f'  → {result}  ({t:.4f}s)')
print(f'Total: {time.clock() - start:.2f}s')
```

- **`timedcall(solve, example)`** — returns `(elapsed_time, result)` (defined in previous lesson)
- 14 examples solved in ~2 seconds total

### Notable results
- `ODD + ODD == EVEN` → `655 + 655 == 1310` (or `855 + 855 == 1710`)
- `A**2 + B**2 == C**2` → Pythagorean triples: `3²+4²=5²`, `5²+12²=13²`
- `AN + BN == CN and N > 1` → works for N=2 (Fermat's Last Theorem says N>2 has no solution!)
- Lowercase letters are **not** substituted — they act as fixed Python identifiers/operators

### Quiz: print all solutions instead of just the first

Change `return f` → `print(f)` in `solve`. The loop continues instead of stopping at the first match.

---

## ⏱️ Profiling with `cProfile`

```bash
# From terminal:
python -m cProfile crypt.py
```

```python
# From within Python:
import cProfile
cProfile.run('solve("ODD + ODD == EVEN")')
```

### Profile results (example — ~75s total run)

| Function | Time (s) | Notes |
|----------|----------|-------|
| `solve` | ~75 | top-level driver |
| `valid` | ~63 | ← **bottleneck** |
| `eval` | ~47 | inside `valid` |
| `re.search` | ~12 | inside `valid` (regex check) |
| `str.maketrans` + `translate` | ~3 | inside `fill_in` |

> Most time is inside `valid` → `eval` is the expensive call.

---

## 📉 Law of Diminishing Returns

> **Optimize where the time actually is.**

If total runtime = 75s and 63s is in `valid`:
- Making `fill_in` infinitely fast saves at most **12s**
- Making `valid` 2× faster saves **~31s**

**Implication:** Focus optimization effort on `valid` (specifically `eval`), not on `fill_in` or other small parts.

$$\text{Max speedup} = \frac{1}{1 - p + \frac{p}{s}} \quad \text{(Amdahl's Law)}$$

Where $p$ = fraction of time in the part you speed up, $s$ = speedup factor for that part.


---

## ⚡ Optimizing the Solver — Fewer `eval` Calls

### Diminishing Returns Quiz

> A=90s, B=10s → total=100s. Speed up B by **10×**. How much faster is the whole program?

Answer: **~1.1× faster** (91s → only 1s saved from B, A unchanged at 90s).

→ **Bottleneck: `eval` inside `valid` (~47/75s = 63% of runtime)**

---

### Strategies: fewer `eval` calls

| Approach | Viable? |
|----------|---------|
| Split formula into smaller evals | ✗ — no clear benefit |
| Fill in one digit at a time (prune early) | ✗ — partial fill (`8V8N`) doesn't eval cleanly |
| **Compile formula to a lambda once, call it N times** | ✓ — best option |

---

### Key idea: compile formula → lambda

Instead of calling `eval` on every permutation, compile the formula **once** into a lambda function, then call that function for each permutation:

```python
# eval parses + generates code every single time:
eval('123 == 45**2')   # repeated N! times → slow

# compile once → call N times:
f = eval('lambda Y, M, E, U, O: (100*U+10*O+1*Y) == (10*E+M)**2')
f(1, 2, 3, 4, 5)   # just executes, no re-parsing → fast
```

---

### `compile_word(word)`

Converts an uppercase word into its polynomial digit expression:

```python
def compile_word(word):
    """compile_word('YOU') => '(100*Y+10*O+1*U)'"""
    if word.isupper():
        terms = [f'{10**i}*{d}' for i, d in enumerate(word[::-1])]
        return '(' + '+'.join(terms) + ')'
    return word
```

- `word[::-1]` → reverse so index 0 = units place
- `enumerate` → pairs `(power, letter)`
- Non-uppercase tokens (operators, digits) returned unchanged

---

### `compile_formula(formula)` + `faster_solve`

```python
import re, itertools, string

def compile_formula(formula, verbose=False):
    """Compile formula string to a lambda. Returns (lambda_fn, letters_str).
    E.g., 'YOU == ME**2' → (lambda Y,O,U,M,E: ..., 'YOUME')"""
    letters = ''.join(set(re.findall('[A-Z]', formula)))
    params = ','.join(letters)
    tokens = map(compile_word, re.split('([A-Z]+)', formula))
    body = ''.join(tokens)
    f = eval(f'lambda {params}: {body}')
    if verbose:
        print(f'lambda {params}: {body}')
    return f, letters

def faster_solve(formula):
    fn, letters = compile_formula(formula)          # compile ONCE
    for digits in itertools.permutations(range(10), len(letters)):
        try:
            if fn(*digits):                          # call lambda — no re-parsing
                table = str.maketrans(letters, ''.join(map(str, digits)))
                return formula.translate(table)
        except ArithmeticError:
            pass
```

- `compile_formula` called **once** — `eval` runs once
- Lambda called up to 10×P(n) times — just executes bytecode, no parsing overhead
- `itertools.permutations(range(10), n)` uses integers directly (no string conversion until a solution is found)

**Result: ~25× faster on simple examples, ~13× on complex ones.**

---

## 📚 Unit Recap — Python Features Used

| Feature | Example / Notes |
|---------|----------------|
| **List comprehensions** | `[x**2 for x in lst if cond]` |
| **Generator expressions** | `(x**2 for x in lst)` — lazy, uses `()` |
| **Generator functions** | `def f(): yield x` — lazy, keeps state |
| **Polymorphism** | `timedcalls(n, ...)` — `n` int → repeat N times, float → repeat for N seconds |
| **`isinstance`** | Used to check type at runtime |
| **`eval`** | Maps string → Python object/result; used to compile formulas |
| **`lambda`** | Anonymous function: `lambda x, y: x + y` |
| **Instrumentation** | `timedcall`, `timedcalls` for timing; `c()` generator for counting |

### On variable naming
- Short names (`c`, `i`, `f`) → OK for short-lived debug/loop vars
- Long descriptive names → for anything that persists or is reused


---

# 📝 Problem Set 2

## PS2.1 — `compile_formula`: Reject Leading Zeros

**Goal:** Modify `compile_formula` so the compiled lambda returns `False` when a **multi-digit word starts with `0`**.

### Rule
- `012` → invalid (leading zero)
- `0` alone → valid (single-digit zero is fine)
- Only the **first letter** of a word with 2+ letters is a "leading digit"

### Strategy

1. Find all **first letters of multi-letter words** using regex:
   ```python
   first_letters = set(re.findall(r'\b([A-Z])[A-Z]', formula))
   ```
2. Build a **guard clause** that checks none of them is `0`:
   ```python
   guard = ' and '.join(f'{L} != 0' for L in first_letters)
   ```
3. Prepend it to the lambda body:
   ```python
   body = f'{guard} and ({original_body})'
   ```

### Modified `compile_formula`

```python
def compile_formula(formula, verbose=False):
    letters = ''.join(set(re.findall('[A-Z]', formula)))
    parms = ', '.join(letters)
    tokens = map(compile_word, re.split('([A-Z]+)', formula))
    body = ''.join(tokens)

    # guard: first letter of any multi-letter word must not be 0
    first_letters = set(re.findall(r'\b([A-Z])[A-Z]', formula))
    if first_letters:
        guard = ' and '.join(f'{L} != 0' for L in sorted(first_letters))
        body = f'{guard} and ({body})'

    f = eval(f'lambda {parms}: {body}')
    if verbose: print(f'lambda {parms}: {body}')
    return f, letters
```

### Example output (verbose)
```
# For 'YOU == ME**2':
lambda Y,O,U,M,E: Y != 0 and M != 0 and ((1*U+10*O+100*Y) == (1*E+10*M)**2)
```

### Key tests
```python
assert faster_solve('A + B == BA') == None   # '1 + 0 == 01' must be rejected
assert faster_solve('YOU == ME**2') in ('289 == 17**2', '576 == 24**2', '841 == 29**2')
assert faster_solve('X / X == X') == '1 / 1 == 1'
```


---

## PS2.2 — Floor Puzzle (Constraint Satisfaction)

**Problem:** 5 residents (Hopper, Kay, Liskov, Perlis, Ritchie) live on floors 1–5. Find who lives where.

### Constraints
| Constraint | Expression |
|-----------|-----------|
| Hopper ≠ top floor | `Hopper != 5` |
| Kay ≠ bottom floor | `Kay != 1` |
| Liskov ≠ top or bottom | `Liskov != 1 and Liskov != 5` |
| Perlis above Kay | `Perlis > Kay` |
| Ritchie not adjacent to Liskov | `abs(Ritchie - Liskov) > 1` |
| Liskov not adjacent to Kay | `abs(Liskov - Kay) > 1` |

### Solution

```python
import itertools

def floor_puzzle():
    floors = range(1, 6)  # [1, 2, 3, 4, 5]
    for (Hopper, Kay, Liskov, Perlis, Ritchie) in itertools.permutations(floors):
        if (Hopper != 5 and
            Kay != 1 and
            Liskov != 1 and Liskov != 5 and
            Perlis > Kay and
            abs(Ritchie - Liskov) > 1 and
            abs(Liskov - Kay) > 1):
            return [Hopper, Kay, Liskov, Perlis, Ritchie]

# Result: [3, 2, 4, 5, 1]
# Hopper=3, Kay=2, Liskov=4, Perlis=5, Ritchie=1
```

### Takeaway
Same pattern as the Zebra Puzzle:
- Assign symbolic values → **brute-force permutations** → filter by constraints
- Readable `if`-chain maps directly to the English problem statement


---

## PS2.3 — Longest Subpalindrome Slice

**Goal:** Return `(i, j)` such that `text[i:j]` is the longest palindrome inside `text`.

### Palindrome rules here
- Case-insensitive: `'Racecar'` is a palindrome
- Spaces/punctuation **count** and must mirror exactly
- Return `(i, j)` indices, not the string itself

---

### ❌ Naive approach — O(N³)

Try every `(i, j)` pair and check if `text[i:j]` is a palindrome.  
Each check costs O(N) → $O(N^2)$ substrings × $O(N)$ check = **O(N³) total**.

---

### ✅ Efficient approach — Grow from center — O(N²)

**Key insight:** A palindrome is symmetric around its center.  
→ Start at a center, expand outward as long as characters match.

Two center types to cover all palindromes:
| Type | Example | Starting position |
|------|---------|-----------------|
| Odd-length | `racecar` (center = `e`) | `(i, i+1)` |
| Even-length | `abba` (center = gap) | `(i, i)` — empty start |

**Why this works:** If `text[i:j]` is a palindrome and `text[i-1] == text[j]`, then `text[i-1:j+1]` is also a palindrome. Each expansion reuses prior knowledge → no redundant comparisons.

---

### Core: `grow(text, start, end)`

```python
def grow(text, start, end):
    """Expand palindrome outward from center [start:end] as far as possible."""
    while (start > 0 and end < len(text) and
           text[start - 1].upper() == text[end].upper()):
        start -= 1
        end += 1
    return (start, end)
```

- `.upper()` handles case-insensitivity
- Stops when characters don't match or edges are reached

---

### `longest_subpalindrome_slice(text)`

```python
def longest_subpalindrome_slice(text):
    if not text: return (0, 0)
    candidates = [grow(text, i, i+d) for i in range(len(text)) for d in [1, 2]]
    return max(candidates, key=lambda ij: ij[1] - ij[0])
```

- `d=1` → odd-length center (single char)
- `d=2` → even-length center (between two chars)
- `max(..., key=lambda ij: ij[1] - ij[0])` → pick the longest slice

---

### Complexity
| Metric | Value |
|--------|-------|
| Centers tried | O(N) |
| Expansions per center | O(N) worst case |
| Character accesses counted | **O(N²)** |

> The efficiency constraint is measured by character accesses (`text[i]` operations), not runtime.

---

### Tests
```python
L = longest_subpalindrome_slice
assert L('racecar')                   == (0, 7)
assert L('Racecar')                   == (0, 7)   # case-insensitive
assert L('RacecarX')                  == (0, 7)
assert L('Race carr')                 == (7, 9)   # 'rr'
assert L('')                          == (0, 0)
assert L('something rac e car going') == (8, 21)
assert L('xxxxx')                     == (0, 5)
assert L('Mad am I ma dam.')          == (0, 15)
```


---

## 🔤 Tool 1: Language — Regular Expressions

A **regular expression (regex)** describes an infinite set of strings with a compact pattern.  
Python module: `re` — key functions: `re.search(pattern, text)`, `re.match(pattern, text)`.

### Special characters (minimal subset)

| Char | Meaning | Example | Matches |
|------|---------|---------|---------|
| `*` | 0 or more of previous | `ba*` | `b`, `ba`, `baa`, … |
| `?` | 0 or 1 of previous | `ba?` | `b`, `ba` |
| `.` | any single character | `b.` | `ba`, `bb`, `b7`, … |
| `^` | start of text | `^b` | strings starting with `b` |
| `$` | end of text | `a$` | strings ending with `a` |

Any non-special character matches itself literally.

---

### Implementing `search` and `match` from scratch

```python
def search(pattern, text):
    """True if pattern appears anywhere in text."""
    if pattern.startswith('^'):
        return match(pattern[1:], text)       # ^ → must start here
    else:
        return match('.*' + pattern, text)    # no ^ → match anywhere

def match(pattern, text):
    """True if pattern matches at the start of text."""
    if pattern == '':
        return True                            # empty pattern matches anything
    elif pattern == '$':
        return (text == '')                    # $ matches only at end
    elif len(pattern) > 1 and pattern[1] in '*?':
        p, op, pat = pattern[0], pattern[1], pattern[2:]
        if op == '*':
            return match_star(p, pat, text)
        elif op == '?':
            return (match1(p, text) and match(pat, text[1:])) or match(pat, text)
    else:
        return match1(pattern[0], text) and match(pattern[1:], text[1:])

def match1(p, text):
    """Match a single character p against the first char of text."""
    if not text: return False
    return p == '.' or p == text[0]

def match_star(p, pattern, text):
    """Match p* (zero or more p) followed by pattern against text."""
    return match(pattern, text) or (match1(p, text) and match_star(p, pattern, text[1:]))
```

### Key design points
- `search` → reduces to `match` (with `.*` prefix or by stripping `^`)
- `match` → recursive; each call consumes one character from pattern and text
- `match_star` → try 0 occurrences first (avoids infinite loop)
- Representation: regex is just a **string** here; more complex cases use **trees**

---

## 🧩 Tool 2: Functions as First-Class Objects

Functions in Python are values — they can be:
- Passed as arguments
- Returned from other functions
- Stored in variables or data structures

This enables powerful patterns used throughout this course:

| Pattern | Example from course |
|---------|-------------------|
| **Higher-order functions** | `timedcall(fn, *args)` — takes `fn` as arg |
| **`eval` to build functions** | `compile_formula` → builds a lambda as a string, then `eval`s it |
| **Generators** | `fill_in(formula)` → `yield` for lazy iteration |
| **`lambda`** | Anonymous one-expression functions: `lambda x: x**2` |

### Why this matters
- **Separation of concerns:** produce data (generator) vs. consume data (solver) vs. check data (validator)
- **Compile once, run many times:** build a lambda once with `eval`, call it millions of times cheaply
- **Composability:** small, pure functions combine cleanly (`match` calls `match1`, `match_star`)


---

## Regex — Tree-Based API & `matchset`

### Why switch from strings to trees?

String regex (`'ba*!'`) is convenient to type but hard to compose programmatically.  
A **function-call API** builds regex as nested structures (trees), which are easier to process recursively.

### The API

| Call | Matches |
|------|---------|
| `lit(s)` | exactly the string `s` |
| `seq(x, y)` | `x` followed by `y` |
| `alt(x, y)` | `x` or `y` |
| `star(x)` | 0 or more `x` |
| `plus(x)` | 1 or more `x` — shorthand for `seq(x, star(x))` |
| `oneof('abc')` | any one of the listed characters |
| `eol` | matches only at end of string |
| `dot` | any single character |

---

### Key insight: represent partial results as a **set of remainders**

Instead of returning `True/False`, `matchset(pattern, text)` returns **the set of all possible remainders** after a match.

- `matchset(lit('a'), 'aaab')` → `{'aab'}` (consumed one `a`)
- `matchset(star(lit('a')), 'aaab')` → `{'aaab', 'aab', 'ab', 'b'}` (consumed 0, 1, 2, or 3 a's)
- Empty set `set()` → **no match**
- Set containing `''` → **matched the whole string**

This makes `seq` simple: match `x`, get remainders, then match `y` against each remainder.

---

### `matchset` implementation

```python
def matchset(pattern, text):
    """Return set of remainders after matching pattern at start of text."""
    op, x, y = pattern[0], pattern[1], pattern[2] if len(pattern) > 2 else None

    if op == 'lit':
        return {text[len(x):]} if text.startswith(x) else set()

    elif op == 'seq':
        return {t2 for t1 in matchset(x, text)
                   for t2 in matchset(y, t1)}

    elif op == 'alt':
        return matchset(x, text) | matchset(y, text)   # union

    elif op == 'dot':
        return {text[1:]} if text else set()

    elif op == 'oneof':
        return {text[1:]} if text and text[0] in x else set()

    elif op == 'eol':
        return {''} if text == '' else set()

    elif op == 'star':
        # 0 matches OR 1 match + recurse
        return ({text} |
                {t2 for t1 in matchset(x, text) if t1 != text
                    for t2 in matchset(pattern, t1)})
```

> **`set()` ≠ `{''}` —** empty set = failure; `{''}` = matched entire string.

---

### `search` and `match` on top of `matchset`

```python
def match(pattern, text):
    """Longest match at start of text, or None."""
    remainders = matchset(pattern, text)
    if remainders:
        shortest = min(remainders, key=len)
        return text[:len(text) - len(shortest)]
    return None

def search(pattern, text):
    """Earliest (leftmost) + longest match anywhere in text, or None."""
    for i in range(len(text) + 1):
        m = match(pattern, text[i:])
        if m is not None:
            return m
    return None
```

- `match` → tries at position 0 only; picks the **longest** match (shortest remainder)
- `search` → slides the window left-to-right, returns the **first** match found


---

## Interpreter → Compiler → Generator

### Interpreter vs Compiler

The **interpreter** (`matchset`) re-parses the pattern tuple on every call. Since the pattern is static, this is wasted work.

The **compiler** approach: each constructor returns a **function** that *is* the pattern — call `pattern(text)` directly.

| | Interpreter | Compiler |
|---|---|---|
| `lit('a')` returns | `('lit', 'a')` tuple | `lambda t: …` function |
| How to call | `matchset(pattern, text)` | `pattern(text)` |

```python
null = frozenset([])

def lit(s):       return lambda t: {t[len(s):]} if t.startswith(s) else null
def seq(x, y):    return lambda t: set().union(*map(y, x(t)))
def alt(x, y):    return lambda t: x(t) | y(t)
def oneof(chars): return lambda t: {t[1:]} if (t and t[0] in chars) else null
dot =             lambda t: {t[1:]} if t else null
eol =             lambda t: {''} if t == '' else null
def star(x):      return lambda t: ({t} | {t2 for t1 in x(t) if t1 != t
                                           for t2 in star(x)(t1)})
def plus(x):   return seq(x, star(x))
def opt(x):    return alt(lit(''), x)
```

> Anything **left of `lambda`** runs once (compile time); **right of `lambda`** runs every call.  
> `match` changes one line: `matchset(pattern, text)` → `pattern(text)`.

### Generator

Generates all strings in a language up to given lengths: `pattern(Ns: set[int]) → set[str]`.

```python
epsilon = lit('')

def lit(s):       return lambda Ns: {s} if len(s) in Ns else null
def alt(x, y):    return lambda Ns: x(Ns) | y(Ns)
def oneof(chars): return lambda Ns: set(chars) if 1 in Ns else null
def opt(x):       return alt(epsilon, x)
def star(x):      return lambda Ns: opt(plus(x))(Ns)
def plus(x):      return lambda Ns: genseq(x, star(x), Ns, startx=1)
def seq(x, y):    return lambda Ns: genseq(x, y, Ns)
```

**`genseq` and infinite loop fix:** `plus(opt(a))` loops forever because `opt` can return `''`, never reducing `Ns`. Fix: `startx=1` in `plus` forces `x` to consume ≥ 1 character → `Ns` shrinks each step.

```python
def genseq(x, y, Ns, startx=0):
    if not Ns: return null
    xmatches = x(set(range(startx, max(Ns) + 1)))
    yNs = {n - len(m1) for n in Ns for m1 in xmatches if n - len(m1) >= 0}
    return {m1 + m2 for m1 in xmatches for m2 in y(yNs) if len(m1)+len(m2) in Ns}
```


---

## Decorators: `n_ary`, `update_wrapper`, `decorator`

### `n_ary` — Extend Binary Functions to N Arguments

`seq(x, y)` only takes 2 args. `n_ary` right-folds: `seq(a, b, c, d)` → `seq(a, seq(b, seq(c, d)))`.

```python
def n_ary(f):
    """Given binary f(x, y), return an n_ary version: f(x,y,z) = f(x, f(y,z)); f(x) = x."""
    def n_ary_f(x, *args):
        return x if not args else f(x, n_ary_f(*args))
    return n_ary_f

# @decorator syntax is sugar for:  seq = n_ary(seq)
@n_ary
def seq(x, y): return ('seq', x, y)
```

### `update_wrapper` and `@decorator`

Without `update_wrapper`, `help(seq)` shows the inner wrapper's name, not `seq`.  
The meta-decorator `decorator` applies `update_wrapper` automatically to every decorator:

```python
from functools import update_wrapper

def decorator(d):
    """Make d a proper decorator: copies __name__ and __doc__ to wrapped fn."""
    def _d(fn):
        return update_wrapper(d(fn), fn)
    update_wrapper(_d, d)
    return _d

decorator = decorator(decorator)   # applies to itself (Darius Bacon one-liner)
```

| `update_wrapper` call | Copies from | Into |
|---|---|---|
| `update_wrapper(d(fn), fn)` | original `fn` | decorated result |
| `update_wrapper(_d, d)` | decorator `d` | wrapper `_d` |


---

## `memo` — Memoization Decorator (Performance Tool)

### Problem: repeated computation in recursion

`fib(30)` without memoization makes **2.6 million** calls.  
With memoization: only **59** calls.

The ratio of `calls(n)` / `calls(n-1)` converges to $\frac{1+\sqrt{5}}{2} \approx 1.618$ — the **Golden Ratio**.

---

### Implementation

```python
@decorator
def memo(f):
    """Decorator: cache results of f; avoids recomputation for same args."""
    cache = {}
    def _f(*args):
        try:
            return cache[args]
        except KeyError:
            cache[args] = result = f(*args)
            return result
        except TypeError:          # args contain unhashable type (e.g. list)
            return f(*args)        # skip cache, call directly
    return _f
```

**Why `try/except` instead of `if args in cache`?**
- Covers 3 cases in one block: hit, miss, unhashable
- Lists (and other mutables) are **unhashable** because hash tables require stable keys — a mutable object's hash could change after insertion

> Mutable → unhashable: if a list `[1,2,3]` is stored under hash `6`, then mutated to `[10,2,3]`, Python would look in slot `15` and never find it.

---

## Decorator Taxonomy

| Decorator | Type | Purpose |
|-----------|------|---------|
| `n_ary` | Expressiveness | Extend binary functions to N args |
| `memo` | Performance | Cache results, avoid redundant calls |
| `countcalls` | Debugging | Count how many times a function is called |
| `trace` | Debugging | Print indented call/return tree |
| `disabled` | Debugging | Turn off a decorator without removing it |

---

## `countcalls` — Count Function Calls

```python
@decorator
def countcalls(f):
    """Count the number of times f is called."""
    def _f(*args):
        countcalls.counts[_f.__name__] = countcalls.counts.get(_f.__name__, 0) + 1
        return f(*args)
    return _f
```

---

## `trace` — Indented Call/Return Tree

```python
@decorator
def trace(f):
    indent = '   '
    def _f(*args):
        signature = '%s(%s)' % (f.__name__, ', '.join(map(repr, args)))
        print('%s--> %s' % (trace.level * indent, signature))
        trace.level += 1
        try:
            result = f(*args)
            print('%s<-- %s == %s' % ((trace.level - 1) * indent, signature, result))
        finally:
            trace.level -= 1    # always restore level, even if exception raised
        return result
    trace.level = 0
    return _f
```

**Key points:**
- `trace.level` stored as function attribute — shared across all calls
- `finally` guarantees the indentation level is always decremented, even on exception
- Shows the recursive call tree visually: `-->` on entry, `<--` on return

**Example output for `fib(3)`:**
```
--> fib(3)
   --> fib(2)
      --> fib(1)
      <-- fib(1) == 1
      --> fib(0)
      <-- fib(0) == 1
   <-- fib(2) == 2
   --> fib(1)
   <-- fib(1) == 1
<-- fib(3) == 3
```

---

## `disabled` — Turn Off a Decorator

```python
disabled = lambda f: f    # identity function — returns f unchanged
```

Usage:
```python
trace = disabled    # all @trace decorators now have zero overhead
```

- No need to remove `@trace` throughout the codebase
- The decorated function is returned as-is — zero overhead at runtime
- `disabled` is not itself a decorator in the traditional sense; it is the **identity function**


---

## Regular vs Context-Free Languages

| | Regular | Context-Free |
|---|---------|-------------|
| Tool | Regex | Grammar + Parser |
| Balanced parens? | ❌ No | ✅ Yes |
| Example | `ba*!` | `E → E + T \| T` |

> **Why can't regex handle balanced parentheses?**  
> Regex can match a fixed nesting depth (1, 2, 3 levels) but **not arbitrary depth**.  
> Recognizing `((…))` with any number of matching pairs requires counting — that requires memory beyond what a finite automaton has.

---

## Wishful Thinking: Designing a Grammar DSL

**Approach:** write the language you *wish* you had, then implement it.

```
Exp   => Term ([+-] Exp)*
       | Term
Term  => Factor ([*/] Term)*
       | Factor
Factor => Num | Var | ( Exp )
Num   => \d+
Var   => [a-zA-Z]\w*
```

This human-readable text is converted into a dict `G`:
- **Keys** = non-terminal names (`Exp`, `Term`, …)
- **Values** = tuple of alternatives, each a list of atoms (right-hand side always `tuple[list]`)

---

## `grammar()` — Text → Dictionary

```python
def grammar(description, whitespace=r'\s*'):
    G = {' ': whitespace}
    description = description.replace('\t', ' ')
    for line in description.split('\n'):
        if '=>' not in line: continue
        lhs, rhs = line.split('=>', 1)
        G[lhs.strip()] = tuple(alt.split() for alt in rhs.split('|'))
    return G
```

- `G[' ']` stores the whitespace regex used as prefix before every token
- Right-hand side: **tuple of lists** — tuple = alternatives, list = sequence of atoms

---

## `parse()` — Recursive Descent Parser

Returns `(tree, remainder)` on success; `Fail = (None, None)` on failure.

| Case | Handling |
|------|----------|
| Symbol in grammar | Try alternatives left-to-right; commit to first success |
| Regex (terminal) | `re.match` with whitespace prefix; advance text here only |
| Sequence (list) | Parse each atom in order; fail if any atom fails |

```python
Fail = (None, None)

def parse(start_symbol, text, grammar):
    tokenizer = grammar[' '] + '(%s)'

    @memo
    def parse_atom(atom, text):
        if atom in grammar:
            for alt in grammar[atom]:
                tree, rem = parse_sequence(alt, text)
                if rem is not None:
                    return [atom] + tree, rem
            return Fail
        else:
            m = re.match(tokenizer % atom, text)
            if not m: return Fail
            return m.group(1), text[m.end():]

    def parse_sequence(sequence, text):
        result = []
        for atom in sequence:
            tree, text = parse_atom(atom, text)
            if text is None: return Fail
            result.append(tree)
        return result, text

    return parse_atom(start_symbol, text)
```

`@memo` on `parse_atom` caches `(atom, text)` pairs → avoids exponential backtracking.

**Order in alternatives matters:** longest/most-specific alternative must come first.

---

## `verify_grammar(G)` — Debugging Tool

| Category | Meaning |
|----------|---------|
| Non-terminals | Keys (left-hand side) |
| Terminals | Right-hand side atoms not in keys (should be regexes) |
| Suspects | Look like names but missing from keys — likely typos |
| Orphans | Defined on left but never referenced on right — unused rules |

---

## Final Summary

| Topic | Key takeaway |
|-------|-------------|
| **Language** | Define the language you want (DSL), not just Python's |
| **Grammar** | Finite description of an infinite language; text → dict |
| **Interpreter** | Walks data structure at runtime; flexible but repeats work |
| **Compiler** | Translates pattern to a function once; call it directly |
| **Parser** | Recognizer + tree builder; returns `(tree, remainder)` |
| **Functions as values** | More composable than statements; do-now vs do-later |
| **Decorators** | `memo` (performance), `trace`/`countcalls` (debug), `n_ary` (expressiveness) |
| **DRY** | One place per logic; compose and reuse instead of copy-paste |


---

## Problem Set 3

### PS3.1 — JSON Grammar Parser

**Goal:** Write a grammar (using the `grammar()` / `parse()` tools from Unit 3) that can parse the full JSON language.

**Key concepts:**
- JSON grammar defined at [json.org](https://json.org) — must be translated to the `Symbol => alt1 | alt2` format
- Top-level symbol: `value`
- Main JSON types to cover:

| Symbol | Description |
|--------|-------------|
| `value` | string \| number \| object \| array \| true \| false \| null |
| `object` | `{ members }` or `{}` |
| `members` | `pair` or `pair , members` (longer alternative first!) |
| `pair` | `string : value` |
| `array` | `[ elements ]` or `[]` |
| `elements` | `value` or `value , elements` |
| `string` | `"..."` — regex `"[^"]*"` |
| `number` | `int frac exp` — use regex to collapse digits |

**Important rules:**
- Put the **longer alternative first** — the parser is a deterministic PEG parser (order matters)
- No left recursion allowed
- Use regex shortcuts to avoid spelling out individual digits:  
  `int => -?[0-9]+`, `frac => \.[0-9]+`, `exp => [eE][+\-]?[0-9]+`

**Usage:**
```python
JSON = grammar("""...""", whitespace=r'\s*')
def json_parse(text): return parse('value', text, JSON)
```

---

### PS3.2 — Efficient Inverse Functions

**Goal:** Implement `inverse(f)` that returns `f⁻¹` in **O(log n)** time instead of O(n).  
Constraint: `f` must be monotonically increasing on non-negative numbers.

**Slow approach (linear):** step forward by `delta` until `f(x) >= y` → too slow for large inputs.

**Fast approach — Double then Binary Search:**

1. **Find bounds** — start at `x=delta`, keep **doubling** `x` until `f(x) >= y`  
   → gives `[low, high]` in O(log y/delta) steps

2. **Binary search** within `[low, high]`:
   - `mid = (low + high) / 2`
   - If `f(mid) > y` → `high = mid`
   - If `f(mid) < y` → `low = mid`
   - If `f(mid) == y` → return `mid`
   - Stop when `high - low <= delta`

**Result:** logarithmic runtime — handles very large inputs (e.g. `10^8`) in milliseconds.

```python
def inverse(f, delta=1/128.):
    def f_1(y):
        # Phase 1: find upper bound by doubling
        high = delta
        while f(high) < y:
            high *= 2
        low = high / 2
        # Phase 2: binary search
        while high - low > delta:
            mid = (low + high) / 2.0
            if f(mid) < y:   low = mid
            elif f(mid) > y: high = mid
            else:            return mid
        return high if (f(high)-y < y-f(low)) else low
    return f_1

sqrt      = inverse(lambda x: x*x)
log10     = inverse(lambda x: 10**x)
cuberoot  = inverse(lambda x: x**3)
```

---

### PS3.3 — Finding HTML Start Tags with Regex

**Goal:** `findtags(text)` → returns list of all HTML **start tags** found in the text.

**What counts as a start tag:**
- `<tagname attr="value" ...>`
- Tag name: `\w+`
- Attributes: `\w+ = "..."` (no quotes inside values assumed)
- Spaces allowed anywhere inside the `< >`
- Self-closing `/>` also valid
- **Not matched:** `</end>`, bare `<` comparisons, malformed tags

**Approach — single regex:**
```python
import re

def findtags(text):
    tag_re = (r'<\s*\w+' +                        # opening < and tag name
              r'(\s+\w+\s*=\s*"[^"]*")*' +        # zero or more attr="val"
              r'\s*/?>')                            # optional / and closing >
    return re.findall(tag_re, text)
```

**Key regex pieces:**

| Pattern | Matches |
|---------|---------|
| `<\s*\w+` | `<` + optional spaces + tag name |
| `\w+\s*=\s*"[^"]*"` | `attr="value"` pair |
| `(...)*` | zero or more attribute pairs |
| `\s*/?>` | optional `/` then `>` |

---

# Unit 4 — Search Problems

## Water Pouring Problem

**Setup:** Two glasses (e.g. 4 oz and 9 oz), unlimited water from a faucet, goal: measure an exact amount (e.g. 6 oz).  
No markings on the glasses — must use precise pour sequences.

**Inventory of concepts:**

| Concept | Description |
|---------|-------------|
| `glass` | has `capacity` and `current level` |
| `state` | tuple of current levels for all glasses — complete world snapshot |
| `goal` | target amount in any glass |
| `actions` | 6 total: fill x, fill y, empty x, empty y, pour x→y, pour y→x |
| `solution` | sequence of actions from start state to goal state |

**Pour action logic (x → y):** stop when **y is full** OR when **x is empty** — whichever comes first.

---

## Search / Exploration

**This is a search problem**, not a fixed-variable combinatorial problem (like cryptarithmetic).  
Sequence length is unknown → complexity is roughly $6^n$ but $n$ is not fixed.

**State space model:**

```
start: (0, 0)
    ↓  [6 possible actions from each state]
frontier  →  states reached but not yet expanded
explored  →  states already expanded (won't revisit)
goal      →  any state where one glass = target amount
```

**Algorithm (generic search loop):**
1. `frontier = {start_state}`
2. Pick a frontier state, apply all 6 actions → get successor states
3. New states go to frontier; already-explored states are ignored
4. Repeat until goal found, or frontier is empty (→ no solution)

---

## Exploration Strategies

| Strategy | Works? | Notes |
|----------|--------|-------|
| **Don't reverse** (avoid immediate A→B→A) | No | Longer loops (A→B→C→A) still possible |
| **Shortest path first** (BFS) | Yes | Finds solution if one exists; expands level by level |
| **Don't re-explore** (track all visited states) | Yes | Most efficient — eliminates all cycles |

**Key insight:** tracking visited states ("don't re-explore") is the correct strategy.  
It guarantees termination and finds the solution in finite time if one exists.

---

## Solving the Water Pouring Problem

**`pour_problem(X, Y, goal, start=(0,0))`**
- `X`, `Y` — glass capacities
- `goal` — target amount (in either glass)
- Returns a **path**: interleaved list of `[state, action, state, action, ...]`

**Key data structures:**
- `explored` — `set` of visited states (avoids cycles)
- `frontier` — ordered `list` of paths; pop from front, append to end (BFS)

**Successors function** — returns `{new_state: action_label}` with 6 entries:
```python
# Fill/Empty
(X, y) via 'fill X'       (0, y) via 'empty X'
(x, Y) via 'fill Y'       (x, 0) via 'empty Y'
# Pour (two cases each direction)
if x + y <= Y:  (0, x+y)         # all of x fits into y
else:           (x+y-Y, Y)       # y fills up, x keeps remainder
```

**Doctest** — standard Python module; embed tests in docstrings as interactive session snippets:
```python
>>> successors(0, 0, 4, 9)
{(0, 9): 'fill Y', (4, 0): 'fill X', (0, 0): 'empty Y'}
```
Run with `print doctest.testmd()` → `TestResults(failed=0, attempted=9)`.

---

## Bridge & Torch Problem

**Setup:** 4 people must cross a dark bridge; only 1–2 people at a time; flashlight required; flashlight must return.  
Each person has a distinct speed: `1, 2, 5, 10` min. When two cross together, time = max of the two.

**State representation:**
```python
state = (here, there, t)
# here, there: frozensets (hashable!) of people + 'light'
# t: total elapsed time
```
- `frozenset` chosen because it is **hashable** (needed for the `explored` set)
- Including `'light'` inside the frozenset simplifies "where is the torch" tracking

**`bsuccessors(state)` — key logic:**
- If `'light' in here`: generate all 1- and 2-person moves `here → there`
  - `new_here = here - {a, b, 'light'}`
  - `new_there = there | {a, b, 'light'}`
  - `t += max(a, b)`; action = `(a, b, '->')`
- If `'light' in there`: generate all 1- and 2-person returns `there → here`
  - action = `(a, b, '<-')`
- Using sets: `{a, b}` when `a == b` → single person crossing handled automatically

**`path_states(path)` / `path_actions(path)`:**
```python
# path = [s0, a0, s1, a1, s2, ...]
path_states  = path[0::2]   # even indices
path_actions = path[1::2]   # odd indices
```

---

## Bridge Search — Optimising for Time (not steps)

In `bridge_problem`, the frontier is **sorted by total elapsed time** (not path length):
```python
frontier.sort(key=lambda path: path[-1][2])  # sort by t in last state
```
This ensures we always expand the fastest path first (analogous to Dijkstra).

**Bug found during development:**
- Copy-paste error in `bsuccessors`: the "light is there" branch was iterating over `here` instead of `there`
- Symptom: person 1 returned from `there` even though they hadn't crossed yet
- Fix: iterate over the correct set in each branch

**Result after fix:** solution `[1,2,5,10]` → total time **17** min  
*(Send 1+2 → 1 back → send 5+10 → 2 back → send 1+2)*  
The naive greedy solution of always sending the fastest person gives **19** min — the search finds the true optimum.

---

## Bridge Search — Fixes & Optimisations

**Fix 1 — Test goal after `pop()`, not before `append()`**
- Checking on push can return a suboptimal path; a cheaper one may still be on the frontier
- Move goal check to right after `frontier.pop(0)` — only then is it guaranteed to be the cheapest

| Option | Works? |
|--------|--------|
| Exhaust full frontier | Yes, but slow |
| Give frontier one more step | Yes, but complex |
| **Test after pop ✓** | Yes — clean and correct |

**Fix 2 — Remove `t` from state**
- `state = (here, there, t)` → same positions with different `t` = different states → `explored` set useless → infinite loops
- Solution: `state = (here, there)` only; store accumulated cost inside the path:
  `path = [state, (action, total_cost), state, (action, total_cost), ...]`

**`bsuccessors2`** — same as before but without `t`:
```python
def bsuccessors2(state):
    here, there = state
    if 'light' in here:
        return {(here-{a,b,'light'}, there|{a,b,'light'}): (a,b,'->') for a in here-{'light'} for b in here-{'light'}}
    else:
        return {(here|{a,b,'light'}, there-{a,b,'light'}): (a,b,'<-') for a in there-{'light'} for b in there-{'light'}}
```

**`path_cost` / `bcost`:**
```python
def path_cost(path): return 0 if len(path) < 3 else path[-2][1]  # last (action, cost)[1]
def bcost(action):   a, b, _ = action; return max(a, b)
```

**Fix 3 — Frontier deduplication**
- If two paths reach the same state, keep only the cheaper one:
```python
def add_to_frontier(frontier, path):
    state = path[-1]
    for i, p in enumerate(frontier):
        if p[-1] == state:
            if path_cost(path) < path_cost(p): frontier[i] = path
            return
    frontier.append(path)
    frontier.sort(key=path_cost)
```

| Problem | Fix |
|---------|-----|
| Premature goal check | Check after `pop()` |
| `t` in state → cycles | Remove `t`; store cost in path |
| Multiple paths to same state | Keep cheapest on frontier |

---

## Towards a Generic Search Function

**Moral of the story:** search is tricky — many edge cases, easy to introduce bugs.

- **Write more tests** — `doctest` sprinkled throughout helps catch regressions early
- **Reuse, don't rewrite** — every problem-specific reimplementation risks new bugs

**Goal:** extract a single generic `search(start, goal_test, successors, cost)` function reusable across any search problem (water pouring, bridge & torch, etc.) by injecting the problem-specific pieces.

---

## Missionaries & Cannibals Problem

**Setup:** 3 missionaries + 3 cannibals on one side; boat carries 1–2 people; goal: move all to the other side.  
**Constraint:** cannibals must never outnumber missionaries on either bank (or missionaries get eaten).

**State:** `(M1, C1, B1, M2, C2, B2)` — counts on each side; 6-tuple needed to generalise to any N.

**`csuccessors(state)` logic:**
- If `C > M > 0` on either side → return `{}` (no successors, invalid state)
- Otherwise generate all moves using **delta vectors**:

```python
deltas = {(2,0,1, -2,0,-1): 'MM', (0,2,1, 0,-2,-1): 'CC', (1,1,1, -1,-1,-1): 'MC',
          (1,0,1, -1,0,-1): 'M',  (0,1,1, 0,-1,-1): 'C'}
# B1>0 → subtract delta (left→right); B2>0 → add delta (right→left)
```
Actions: `'MM->'`, `'CC->'`, `'MC->'`, `'M->'`, `'C->'` and their `<-` reverses.

---

## Generic `shortest_path_search`

**Inputs:** `start`, `successors(state) → {state: action}`, `is_goal(state) → bool`  
**Output:** shortest path (fewest steps) as `[state, action, state, ...]`

```python
def shortest_path_search(start, successors, is_goal):
    if is_goal(start): return [start]
    explored, frontier = set(), [[start]]
    while frontier:
        path = frontier.pop(0)
        s = path[-1]
        for state, action in successors(s).items():
            if state not in explored:
                explored.add(state)
                path2 = path + [action, state]
                if is_goal(state): return path2
                frontier.append(path2)
    return []
```

`mc_problem2` then becomes a single call:
```python
def mc_problem2(start=(3,3,1,0,0,0), goal=None):
    if goal is None: goal = (0,0,0) + start[:3]
    return shortest_path_search(start, csuccessors, lambda s: s == goal)
```

---

## Generic `lowest_cost_search`

**Inputs:** `start`, `successors`, `is_goal`, `action_cost(action) → number`  
**Output:** lowest-cost path as `[state, (action, total_cost), state, ...]`

Key differences from `shortest_path_search`:
- Path stores `(action, total_cost)` tuples instead of bare actions
- Frontier sorted by `path_cost` (total accumulated cost), not length
- `add_to_frontier` deduplicates: keeps cheapest path per end-state

```python
def lowest_cost_search(start, successors, is_goal, action_cost):
    explored, frontier = set(), [[start]]
    while frontier:
        path = frontier.pop(0)
        state1 = path[-1]
        if is_goal(state1): return path
        explored.add(state1)
        pcost = path_cost(path)
        for state, action in successors(state1).items():
            if state not in explored:
                path2 = path + [(action, pcost + action_cost(action)), state]
                add_to_frontier(frontier, path2)
    return []
```

`bridge_problem3` becomes:
```python
def bridge_problem3(here):
    here = frozenset(here) | {'light'}
    start = (here, frozenset())
    return lowest_cost_search(start, bsuccessors2,
                              lambda s: not s[0] or s[0] == {'light'},
                              bcost)
```

---

## Unit 4 — Key Takeaways

| Concept | Summary |
|---------|---------|
| **Search problems** | Unknown-length sequences; complexity ≈ $b^n$ where $b$ = branching factor |
| **`shortest_path_search`** | BFS; optimises number of steps; needs `start`, `successors`, `is_goal` |
| **`lowest_cost_search`** | Dijkstra-like; optimises cumulative cost; needs `action_cost` too |
| **State design** | Keep only what's needed; avoid including mutable/time-varying fields like `t` |
| **Avoid bugs** | Write many tests; reuse generalised search tools instead of rewriting |

---

# Problem Set 4

## PS4-1 — Refactoring `bsuccessors3`: New State Representation

**Motivation:** The previous representation mixed people and the light inside the same set, requiring special-case logic everywhere.

**New representation:**

| Element | Type | Meaning |
|---------|------|---------|
| `here` | `frozenset` | People currently on the starting side |
| `there` | `frozenset` | People currently on the destination side |
| `light` | `int` (0 or 1) | `0` = torch on *here* side, `1` = torch on *there* side |

**State tuple:** `(here, there, light)` — light is no longer in the sets.

**Action tuple:** `(frozenset_of_travelers, '→')` or `(frozenset_of_travelers, '←')` — no light in the action either.

**`bsuccessors3` solution:**

```python
def bsuccessors3(state):
    here, there, light = state
    if light == 0:  # torch is on 'here' side
        return {(here - {a,b}, there | {a,b}, 1): (frozenset([a,b]), '->')
                for a in here for b in here}
    else:           # torch is on 'there' side
        return {(here | {a,b}, there - {a,b}, 0): (frozenset([a,b]), '<-')
                for a in there for b in there}
```

**Why simpler:**  
- No need to check `'light' in here` (it was a string sentinel before)
- Index `0` or `1` cleanly selects which side the torch is on
- Actions describe only the travelers, not the light

**Helper `single_successor(state, a, b)`** (optional decomposition):
```python
def single_successor(state, a, b):
    here, there, light = state
    start, dest = (here, there) if light == 0 else (there, here)
    new_light = 1 - light
    new_start = start - {a, b}
    new_dest  = dest  | {a, b}
    if light == 0:
        return (new_start, new_dest, new_light)
    else:
        return (new_dest, new_start, new_light)
```

---

## PS4-2 — `more_pour_problem`: Arbitrary Number of Glasses

**Problem:** generalize the water-pouring problem to any number of glasses with arbitrary capacities.

**Inputs:**
- `capacities` — tuple of glass volumes, e.g. `(1, 2, 4, 8)`
- `goal` — integer target volume to reach in *any* glass
- `start` — optional initial fill levels (defaults to all zeros)

**Output:** path `[state, action, state, action, ...]`

**Actions:**

| Action | Meaning |
|--------|---------|
| `('fill', i)` | Fill glass `i` to capacity |
| `('empty', i)` | Empty glass `i` |
| `('pour', i, j)` | Pour from glass `i` into glass `j` |

**Glass numbering:** 0-indexed — glass `0` is `capacities[0]`, etc.

---

### Key helper: `replace(sequence, i, val)`

Tuples are **immutable** — can't do `state[i] = x`.  
`replace` converts to list, mutates, then reconstructs the original type:

```python
def replace(sequence, i, val):
    s = list(sequence)
    s[i] = val
    return type(sequence)(s)  # works for tuple, list, or str
```

---

### Successor function

```python
def pour_successors(state, capacities):
    succ = {}
    for i in range(len(state)):
        # fill
        succ[replace(state, i, capacities[i])] = ('fill', i)
        # empty
        succ[replace(state, i, 0)] = ('empty', i)
        # pour i → j
        for j in range(len(state)):
            if i != j:
                amount = min(state[i], capacities[j] - state[j])
                s = replace(state, i, state[i] - amount)
                s = replace(s,     j, state[j] + amount)
                succ[s] = ('pour', i, j)
    return succ
```

---

### Full solution

```python
def more_pour_problem(capacities, goal, start=None):
    if start is None:
        start = (0,) * len(capacities)
    return shortest_path_search(
        start,
        lambda s: pour_successors(s, capacities),
        lambda s: goal in s
    )
```

**Key points:**
- `is_goal` just checks `goal in s` — goal reached if *any* glass holds that amount
- The successor closure captures `capacities` — clean use of Python closures
- Reuses `shortest_path_search` unchanged

---

### Test cases

```python
assert more_pour_problem((1, 2, 4, 8), 4) == [
    (0,0,0,0), ('fill',2), (0,0,4,0)]

assert more_pour_problem((1, 2, 4, 8), 5) == [
    (0,0,0,0), ('fill',3), (0,0,0,8),
    ('pour',3,0), (1,0,0,7), ('pour',3,1), (1,2,0,5)]

# Odd goals — impossible with all-even capacities
assert not any(more_pour_problem((8,12,16,20,24), odd) for odd in (3,5,7,9))

# All values 0-27 reachable with powers of 3
assert all(more_pour_problem((1,3,9,27), n) for n in range(28))
assert more_pour_problem((1,3,9,27), 28) == []  # impossible
```

---

## PS4-3 — Subway Route Planning

**Problem:** model a subway map and find shortest routes between stations.

**Goal functions to implement:**

| Function | Description |
|----------|-------------|
| `subway(**lines)` | Build a graph from named lines |
| `ride(here, there, system)` | Find shortest path between two stations |
| `longest_ride(system)` | Find the longest of all shortest paths |

---

### `subway(**lines)` — Building the Graph

**`**lines` (keyword argument unpacking):** captures any number of `name='stop1 stop2 ...'` arguments into a dict.

**Output:** `{station: {neighbor: line_name, ...}, ...}`  
Example: `'foresthills': {'backbay': 'orange'}`

**Strategy:** use `collections.defaultdict(dict)` to avoid key-existence checks, then for each consecutive pair of stops on a line add bidirectional edges:

```python
import collections

def subway(**lines):
    successors = collections.defaultdict(dict)
    for linename, stops in lines.items():
        stops_list = stops.split()
        for a, b in overlapping_pairs(stops_list):
            successors[a][b] = linename
            successors[b][a] = linename
    return successors

def overlapping_pairs(seq):
    return [(seq[i], seq[i+1]) for i in range(len(seq)-1)]
```

---

### `ride(here, there, system)` — Shortest Route

States = station names (strings); actions = line names (strings).  
The `system` dict is already the successor function:

```python
def ride(here, there, system=boston):
    return shortest_path_search(here, system, lambda s: s == there)
```

Example result:
```python
ride('mit', 'government')
# → ['mit', 'red', 'charles', 'red', 'park', 'green', 'government']
```

---

### `longest_ride(system)` — Diameter of the Graph

```python
def longest_ride(system):
    stops = set(s for neighbors in system.values() for s in neighbors)
    return max((ride(a, b, system) for a in stops for b in stops),
               key=lambda path: len(path))
```

**Why `len(path)` not `len(path_states(path))`:** both work; longer path ↔ more states ↔ more steps.

---

### Boston Subway Map

```python
boston = subway(
    blue  ='bowdoin government state aquarium maverick airport suffolk revere wonderland',
    orange='oakgrove sullivan haymarket state downtown chinatown tufts backbay foresthills',
    green ='lechmere science north haymarket government park copley kenmore newton riverside',
    red   ='alewife davis porter harvard central mit charles park downtown south umass mattapan'
)
```

Note: `state`, `park`, `downtown`, `government`, `haymarket` appear on multiple lines — they are **transfer stations**.

---

### Key Takeaways

| Concept | Lesson |
|---------|--------|
| `**kwargs` | Collect arbitrary keyword args into a dict — useful for named-line input |
| `defaultdict(dict)` | Builds nested dicts cleanly without `setdefault` boilerplate |
| Graph as successor dict | `{station: {neighbor: action}}` is directly usable by `shortest_path_search` |
| Reuse search functions | `ride` and `longest_ride` delegate entirely to `shortest_path_search` |
| `overlapping_pairs` | Small utility; useful whenever you need consecutive-element pairs from a sequence |